<b>General pipeline</b>: preprocessing => segmentation => feature extraction => classification (https://peerj.com/articles/cs-620/#fig-5)

<h2><b> IDEAS FROM ARTICLE WITH COMPARISON OF METHODS </b></h2>
Quality improvement: Contrast stretching, Grayscale stretching, Log transformation, Gamma correction,
Image negative, Histogram equalization methods, Adaptive local contrast stretching

Filtering (Gaussian, Poisson, and Quantum noise are different types of
noise artifacts - which one do we have?? because 'If we try to minimize one
class of noise, it may disrupt the other'): Average filter,
Bilateral filter, Laplacian filter, Homomorphic filter, and Butterworth filter, Median
Gaussian filter, and Weiner filter, Median (reduces boundaries), Gaussian (reduces picture information)

Segmentation: segmentation of panoramic X-rays using wavelet transformation shows
better results than adaptive and iterative thresholding, template matching technique, Otsu’s threshold combined with morphological dilation, (gap valley extraction, modified canny
edge detector, guided iterative contour tracing, and template matching), contour-based segmentation,  horizontal integral projection, computing moments and statistical characteristics, edge segmentation methods: Canny
and Sobel, Quantum Particle Swarm Optimization (QPSO)  employed for
multilevel thresholding,  Gaussian kernel-based conditional spatial
fuzzy c-means (GK-csFCM) clustering algorithm.

Classic ML: Feature extraction (Projected principal edge distribution
(PPED) + Geometric properties +
Region descriptors) + SVM, Segmentation of mandibular teeth carried
out by applying Random forest regression-
voting constrained local model (RFRV-
CLM) in two steps: The 1st step gives an
estimate of individual teeth and mandible
regions used to initialize search for the
tooth. In the second step, the investigation
is carried out separately for each tooth.


<h2><b> SUGGESTION AFTER CONSULTING WITH SOME EXPERTS IN ALL TOPICS :D </b></h2>


1. preprocessing:
- CLAHE +
- every noise reduction that do not blur edges (unsharp masking / bilateral filtering / non-local mean)
- morphological operation (erosion / dilatation etc. - check before or after first segmentation)
- validate visually + check strength of found edges (Sobel/Canny)
- Gamma correction
-  Log transformation
- Median filter, Gaussian filter, Wiener filter (check what type of noise there is primarily [SaltNPepper / Gaussian / Photon(Quantum) / ... ]
- contrast stretching
- adaptive thresholding
- averaging (maybe images without segments that have teeth in them, to find some pattern of the noise, that we can possibly remove)
- top-hat (white / black)
- FFT where high value -> set to zero
- Log Gabor [for Speckle noise]
2. segmentation:
- watershed with markers on each tooth
- active contours (snake etc)
- teeth touching / overlapping = concavity analysis + find ways to count teeth properly (opening / closure)
- Validate segmentation (compute Dice coefficient vs ground truth), IoU, visually how they look compared to manually prepared, check number of teeth
- remove very small/large segments
- Laplacian filter for edges
- Otsu's threshold + morphological dilation
- Contour-based segmentation (snake after initialization of predicted space that our teeth should be / level-set)
- Wavelet transformation
3. Feature extraction:
- hu moments, aspect ratio, solidity, circularity/compactness, eccentricity of fitted ellipse
- Position/context features: centroid position, orientation angle, number of neighbors and distances to them, relative position in dental arch
- Texture features (often overlooked but useful): Local Binary Patterns (LBP) on tooth region, Gray-level statistic
- plot how clusterization works (of types / sides / jaws)
- Projected Principal Edge Distribution + Shape Descriptors + Region descriptors (texture etc.) => SVM
4. Classification:
- two-step: tooth type classification (SVM / RF with extracted features)
- second: get specific tooth class (have type, have orientation [bottom/top jaw, left/right part of mouth])
- spatial graph of teeth, compare to template
- cross - validation
- we can even try to extract as many features to try to just generate one class (1..32) for every tooth (easier if we want to see some results)
- graph data: centroid position, tooth type, jaw (top/bottom), neighbors (teeth within certain distance)
- prepare template from all images (average / median positions by teeth type+class, distances, etc.)
- Random Forest estimates approximate regions for each tooth (search regions) => CLM (Constrained Local Model) searches for a tooth in every region
5. Missing teeth:
- check distances between neighbours
- compare to template to see where the tooth should be if dist > expected, mark as missing
-

In [5]:
import numpy as np
import cv2
import json
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import kagglehub
from scipy import ndimage

/tmp/ipykernel_63181/265722106.py:8: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.2)
  from scipy import ndimage


In [6]:
# Load images

def get_images(download_dataset_flag):
    if download_dataset_flag:
        dataset_path = kagglehub.dataset_download("humansintheloop/teeth-segmentation-on-dental-x-ray-images")
        sources = {k: Path(dataset_path) / f'Teeth Segmentation {k}' for k in ['JSON', 'PNG']}
    else:
        dataset_root = Path(r"")
        sources = {k: dataset_root / f'./Teeth Segmentation {k}' for k in ['JSON', 'PNG']}

    meta = {}
    for p in sources.values():
        meta.update(json.loads((p / 'meta.json').read_text()))

    images = [
        (Image.open(img_path), img_path)
        for p in sources.values()
        for img_path in (p / 'd2' / 'img').glob('*')
    ]

    images_np = [(np.array(image), str(p)) for (image, p) in images[:len(images)//2]]
    return images_np

In [7]:
DOWNLOAD_DATASET = True

images_np = get_images(DOWNLOAD_DATASET)

KeyboardInterrupt: 

<h1><b>STEP1:</b> Normalize images, resize or truncate to one size </h1>

In [ ]:
# Get general info about images size

shapes = []
for image, p in images_np:
    shapes.append(image.shape)

min_width = np.min(np.array(shapes)[:, 1])
max_width = np.max(np.array(shapes)[:, 1])
min_height = np.min(np.array(shapes)[:, 0])
max_height = np.max(np.array(shapes)[:, 0])
avg_width = np.median(np.array(shapes)[:, 1])
avg_height = np.median(np.array(shapes)[:, 0])

print(min_width, max_width, min_height, max_height, avg_width, avg_height)


1394 2045 1024 1024 2041.0 1024.0


In [ ]:
# Method changes image size and calculates new polygon positions (but do not save them so we need to either run it every time of rewrite it to save new data

def normalize_image_size(image, image_metadata, output_height=1024, output_width=2045):
    image_height, image_width = image.shape

    scale_y = output_height / image_height
    scale_x = output_width / image_width

    resized_image = cv2.resize(image, (output_width, output_height))
    image_metadata['size'] = {
        'height': output_height,
        'width': output_width,
    }

    for segment_no in range(len(image_metadata['objects'])):
        for vertex_no in range(len(image_metadata['objects'][segment_no]['points']['exterior'])):
            x = image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][0]
            y = image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][1]
            image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][0] = int(x*scale_x)
            image_metadata['objects'][segment_no]['points']['exterior'][vertex_no][1] = int(y*scale_y)

    return resized_image, image_metadata

In [ ]:
import copy

# Method transforms image_number raw images and metadata to resized images and metadata
def adjust_images(images_with_paths, image_number=None):
    normalized_images = []
    images_with_metadata = []
    for image, image_path in images_with_paths[:image_number if image_number else len(images_with_paths)]:
        metadata_filename = image_path.split('\\')[-1] + '.json'
        metadata_path = '\\'.join(image_path.split('\\')[:-2]) + '\\ann\\' + metadata_filename
        with open(metadata_path, 'r') as f:
            image_metadata = json.load(f)
        images_with_metadata.append((image, image_metadata, image_path))
        resized_image, resized_image_metadata = normalize_image_size(image, copy.deepcopy(image_metadata))
        normalized_images.append((resized_image, resized_image_metadata))

    return normalized_images, images_with_metadata

In [ ]:
def get_image_names(images_with_paths):
    return [p.split('\\')[-1] for _, p in images_with_paths]

In [ ]:
resized_images_np, original_images_np = adjust_images(images_np)

In [ ]:
# Compare how image and shapes created from metadata look before and after resize

def display_images_size_compared():
    i = 1
    ipo = 20
    for (resized_image, resized_metadata), (original_image, metadata, path) in zip(resized_images_np[i*ipo: (i+1)*ipo], original_images_np[i*ipo: (i+1)*ipo]):
        if original_image.shape[1] < 2100:
            print(path)
            original_xs = []
            original_ys = []
            for segment_no in range(len(metadata['objects'])):
                arr = np.array(metadata['objects'][segment_no]['points']['exterior'])
                arr = np.concatenate((arr, np.array(metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
                original_xs.append(arr[:, 0])
                original_ys.append(arr[:, 1])

            resized_xs = []
            resized_ys = []
            for segment_no in range(len(resized_metadata['objects'])):
                arr = np.array(resized_metadata['objects'][segment_no]['points']['exterior'])
                arr = np.concatenate((arr, np.array(resized_metadata['objects'][segment_no]['points']['exterior'][0]).reshape(1,2)))
                resized_xs.append(arr[:, 0])
                resized_ys.append(arr[:, 1])

            fig, axes = plt.subplots(1, 2, figsize=(20, 10))
            axes[1].imshow(resized_image, cmap='gray')
            axes[1].set_title(f'Resized image {resized_image.shape[0]}x{resized_image.shape[1]}')
            for x, y in zip(resized_xs, resized_ys):
                axes[1].plot(x, y)
            axes[1].axis('off')

            axes[0].imshow(original_image, cmap='gray')
            axes[0].set_title(f'Original image {original_image.shape[0]}x{original_image.shape[1]}')
            for x, y in zip(original_xs, original_ys):
                axes[0].plot(x, y)
            axes[0].axis('off')

            plt.show()

In [ ]:
display_images_size_compared()

<h1><b>STEP2:</b> Find out what type of noise is present on the images </h1>

https://scikit-image.org/docs/stable/api/skimage.restoration.html#skimage.restoration.estimate_sigma

In [ ]:
# TODO: its not crucial I think, but it would be nice to have to say / write some words about what are the biggest issues with our data quality
def get_noise_report(images):
    pass

<h1><b>STEP3:</b> Image denoising </h1>

In [ ]:
def apply_gamma_correction(images):
    gamma = 2
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255
                      for i in range(256)]).astype("uint8")
    improved_images = []
    for image in images:
        # maybe gamma correction? it looks promising with gamma=2 but you can try something different
        fig, axes = plt.subplots(1, 2, figsize=(20, 20))
        axes[0].imshow(image, cmap='gray')
        axes[0].set_title(f'Original')
        axes[0].axis('off')

        image_enhanced = cv2.LUT(image, table)
        improved_images.append(image_enhanced)

        axes[1].imshow(image_enhanced, cmap='gray')
        axes[1].set_title(f'After gamma correction (gamma = {gamma})')
        axes[1].axis('off')

        plt.show()
        plt.close()

        # Code for comparing gammas
        # fig, axes = plt.subplots(2, 2, figsize=(20, 10))
        # axes[0,0].imshow(image, cmap='gray')
        # axes[0,0].set_title(f'Original')
        # axes[0,0].axis('off')
        # next_axes = [0,1]
        # for gamma in [0.5, 2, 3]:
        #     inv_gamma = 1.0 / gamma
        #     table = np.array([((i / 255.0) ** inv_gamma) * 255
        #                       for i in range(256)]).astype("uint8")
        #
        #     image_enhanced = cv2.LUT(image, table)
        #
        #     axes[next_axes[0], next_axes[1]].imshow(image_enhanced, cmap='gray')
        #     axes[next_axes[0], next_axes[1]].set_title(f'After gamma correction (gamma={gamma})')
        #     axes[next_axes[0], next_axes[1]].axis('off')
        #
        #     next_axes[1] = next_axes[1] + 1
        #     if next_axes[1] == 2:
        #         next_axes = [next_axes[0]+1, 0]
        #
        # plt.show()
        # plt.close()
    return improved_images

In [ ]:
# Images after gamma correction
improved_images_np2 = apply_gamma_correction([img for img, _ in resized_images_np][:5])

In [ ]:
# Gamma correction after CLAHE (too much I think, but you can try

improved_images_np3 = apply_gamma_correction(improved_images_np[:5])

In [ ]:
def best_preprocessing(images):
    gamma = 2.0

    clip_limit = 4
    tile_grid_size = 8

    d = 4
    sigma_color = 50
    sigma_space = 50

    improved_images = []
    for image in images:
        # fig, axes = plt.subplots(1, 2, figsize=(20, 20))
        # axes[0].imshow(image, cmap='gray')
        # axes[0].set_title(f'Original')
        # axes[0].axis('off')

        # 1. Gamma correction - brightens the teeth
        table = np.array([(i / 255.0) ** (1.0 / gamma) * 255 for i in range(256)]).astype("uint8")
        image_filtered = cv2.LUT(image, table)

        # 2. CLAHE – enhances local contrast
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid_size, tile_grid_size))
        image_filtered = clahe.apply(image_filtered)

        # 3. Bilateral filter - removes noise, preserves edges
        image_filtered = cv2.bilateralFilter(image_filtered, d=d, sigmaColor=sigma_color, sigmaSpace=sigma_space)

        # Bilateral filter - removes noise, preserves edges
        # blurred = cv2.GaussianBlur(image_filtered, (0,0), 2)
        # image_filtered = cv2.addWeighted(image_filtered, 1.5, blurred, -0.5, 0)

        # inv_gamma = 1.0 / 2.0
        # table = np.array([ (i / 255.0) ** inv_gamma * 255 for i in range(256) ]).astype("uint8")
        # image_filtered = cv2.LUT(image, table)
        #
        # # 2. CLAHE with higher clipLimit
        # clahe = cv2.createCLAHE(clipLimit=10.0, tileGridSize=(12,12))
        # image_filtered = clahe.apply(image_filtered)
        #
        # # 3. Bilateral - very strong but safe
        # image_filtered = cv2.bilateralFilter(image_filtered, d=10, sigmaColor=80, sigmaSpace=80)

        # 4. Sharpening (unsharp mask)
        # blurred = cv2.GaussianBlur(image_filtered, (0,0), 3.0)
        # image_filtered = cv2.addWeighted(image_filtered, 2.0, blurred, -1.0, 0)

        improved_images.append(image_filtered)

        # axes[1].imshow(image_filtered, cmap='gray')
        # axes[1].set_title(f'After bilateral filter (d={d}, sigmaColor={sigma_color}, sigmaSpace={sigma_space})')
        # axes[1].axis('off')

        plt.show()
        plt.close()

    return improved_images

In [ ]:
preprocessed_images = best_preprocessing([img for img, _ in resized_images_np][:5])

In [ ]:
def preprocess_for_segmentation(images):
    improved_images = []
    for image in images:
        # 1. Noise removal while preserving edges
        denoised = cv2.bilateralFilter(image, 9, 75, 75)

        # 2. Local contrast enhancement (CLAHE)
        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(16, 16))
        enhanced = clahe.apply(denoised)

        # 3. Optional: Top-hat to remove background streaks
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (50, 50))
        final = cv2.morphologyEx(enhanced, cv2.MORPH_TOPHAT, kernel)

        improved_images.append(final)

    return improved_images

In [ ]:
resized_images_np, original_images_np = adjust_images(images_np)
paths = get_image_names(images_np)

In [ ]:
test_images_numbers = [4, 12, 34, 70, 77, 79]
test_images_names = [str(image_number)+'.jpg' for image_number in test_images_numbers]

In [ ]:
test_images_idxs = np.argwhere(np.isin(paths, test_images_names)).reshape(-1)

In [ ]:
def detect_jaw_hybrid(image):
    h, w = image.shape

    # STEP 1: Enhance dark regions
    clahe = cv2.createCLAHE(clipLimit=15.0, tileGridSize=(4,4))
    enhanced = clahe.apply(image)
    if debug:
        plt.imshow(enhanced, cmap='gray')
        plt.show()

    binary = cv2.adaptiveThreshold(
        enhanced,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=101,  # Size of neighborhood (must be odd)
        C=0  # Constant subtracted from mean
    )
    if debug:
        plt.imshow(binary, cmap='gray')
        plt.show()

    edges2 = cv2.Canny(binary, threshold1=120, threshold2=150)

    if debug:
        plt.imshow(edges2, cmap='gray')
        plt.show()

        # BASE FOR THE SEGMENTATION - EDGES OF THE TEETH LOOKS PROMISING

In [ ]:
for img, _ in resized_images_np[:2]:
    debug = True
    detect_jaw_hybrid(img)

<h1><b>STEP4:</b> Segmentation </h1>

Some better images (in terms of how the teeth look): 12.jpg, 4.jpg, 34.jpg

Some images with a few separated teeth: 70.jpg, 77.jpg, 79.jpg

In [ ]:
def preprocess_dental_xray(images):
    improved_images = []
    for image in images:
        preprocessed_img = image.copy()

        # 1. Bilateral Filtration - removes noise, preserves edges
        # d=9 - diameter of pixel neighborhood, sigmaColor/Space=75 - filtering strength
        denoised = cv2.bilateralFilter(preprocessed_img, 9, 80, 80)

        # 2. Morphological Top-Hat Transformation (White-Hat)
        # Used to extract bright objects (teeth) on a darker background.
        # This helps to mitigate the impact of broad light streaks.
        # Kernel size must be close to the width of a single tooth (in pixels).
        kernel_size = (80, 50)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
        tophat = cv2.morphologyEx(denoised, cv2.MORPH_TOPHAT, kernel)

        tophat = cv2.normalize(tophat, None, 0, 255, cv2.NORM_MINMAX)

        # 3. CLAHE (Contrast Limited Adaptive Histogram Equalization)
        # Improves local contrast on the image after Top-Hat
        clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(tophat)

        # 4. (Optional) Image sharpening (Unsharp Masking)
        # gaussian = cv2.GaussianBlur(enhanced, (0, 0), 3.0)
        # unsharp_image = cv2.addWeighted(enhanced, 1.5, gaussian, -0.5, 0, enhanced)

        improved_images.append(enhanced)

    return improved_images

In [ ]:
def segment_teeth_watershed(preprocessed_img):
    """
    Segments individual teeth using marker-controlled watershed.
    Returns binary mask, contours, and visualization image.
    """
    # fig, axes = plt.subplots(1, 3, figsize=(24, 16))

    # Step 1: Otsu thresholding
    _, binary = cv2.threshold(preprocessed_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # axes[0].imshow(binary, cmap='gray')
    # axes[0].set_title('After threshold')

    # Step 2: Clean up binary mask
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel_close)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel_open)

    # axes[1].imshow(binary, cmap='gray')
    # axes[1].set_title('After clean up binary mask')

    # Step 3: Sure background (dilated mask)
    sure_bg = cv2.dilate(binary, kernel_close, iterations=3)

    # axes[2].imshow(sure_bg, cmap='gray')
    # axes[2].set_title('Dilated mask')
    # plt.show()

    # Step 4: Distance transform to find sure foreground (inside teeth)
    dist_transform = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(dist_transform, 0.35 * dist_transform.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)

    # Step 5: Unknown region (between fg and bg)
    unknown = cv2.subtract(sure_bg, sure_fg)

    # Step 6: Create markers
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0

    # Step 7: Apply watershed
    color_img = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
    cv2.watershed(color_img, markers)

    # Step 8: Create final tooth mask and contours
    teeth_mask = np.zeros_like(preprocessed_img)
    teeth_mask[markers > 1] = 255

    contours, _ = cv2.findContours(teeth_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # filtered_contours = []
    # for cnt in contours:
    #     area = cv2.contourArea(cnt)
    #     if area < 100:
    #         continue
    #
    #     x, y, cw, ch = cv2.boundingRect(cnt)
    #     aspect_ratio = cw / ch if ch > 0 else 0
    #
    #     if aspect_ratio < 2.5 and y + ch > preprocessed_img.shape[0] * 0.3:
    #         filtered_contours.append(cnt)
    #
    # return teeth_mask, filtered_contours, markers

    return teeth_mask, contours, markers

In [ ]:
debug = False

In [ ]:
def extract_jaw_directional_edges(image):
    """
    Keep only edges that point outward (jaw boundary)
    Discard inward-facing edges (internal structure)
    """
    clahe = cv2.createCLAHE(clipLimit=15.0, tileGridSize=(4,4))
    enhanced = clahe.apply(image)
    if debug:
        plt.imshow(enhanced, cmap='gray')
        plt.show()
    # STEP 1: Low-threshold Canny
    blurred = cv2.GaussianBlur(enhanced, (3,3), 0)
    edges = cv2.Canny(blurred, threshold1=0, threshold2=20)
    if debug:
        plt.imshow(edges, cmap='gray')
        plt.show()

    # STEP 2: Compute edge directions using Sobel
    sobelx = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=11)
    sobely = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=11)

    # Edge direction (angle)
    edge_angle = np.arctan2(sobely, sobelx)

    # STEP 3: Image center (approximate jaw center)
    h, w = image.shape
    center_x, center_y = w // 2, h // 2

    # STEP 4: For each edge pixel, check if it points outward
    filtered_edges = np.zeros_like(edges)

    edge_coords = np.argwhere(edges > 0)

    for y, x in edge_coords:
        # Vector from center to this pixel
        dx = x - center_x
        dy = y - center_y

        # Angle from center to pixel
        pixel_angle = np.arctan2(dy, dx)

        # Edge gradient angle at this pixel
        gradient_angle = edge_angle[y, x]

        # If gradient points roughly outward, keep it
        angle_diff = abs(pixel_angle - gradient_angle)

        # Allow 90-degree tolerance (perpendicular is also okay)
        if angle_diff < np.pi / 2 or angle_diff > 3 * np.pi / 2:
            filtered_edges[y, x] = 255

    if debug:
        plt.imshow(filtered_edges, cmap='gray')
        plt.show()

    return filtered_edges

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt  # optional - for debugging

def find_upper_mandible_border(idx, image, original):
    h, w = image.shape

    # Optional: slightly blur to reduce noise
    image = cv2.GaussianBlur(image, (5,5), 0)
    if debug:
        plt.imshow(image, cmap='gray')
        plt.show()

    # Focus on middle ~60-70% of image width (avoid ramus/condyle areas)
    margin = int(w * 0.25)
    roi = image[:, margin:-margin]
    if debug:
        plt.imshow(roi, cmap='gray')
        plt.show()

    # Compute horizontal projection (sum down columns)
    proj = np.sum(roi, axis=1).astype(np.float32)

    # Optional: smooth the projection
    proj = cv2.GaussianBlur(proj.reshape(-1,1), (1,51), 0).flatten()

    # We look for big intensity change (drop/rise) in upper-middle part
    search_from = int(h * 0.02)   # skip very top (labels, noise)
    search_to   = int(h*0.35)   # usually mandible starts before middle

    v=10
    ys = []
    diffs = []
    for y in range(search_from+v, search_to):
        diff = proj[y-v:y].mean() - proj[y:y+v].mean()  # drop after bright bone
        ys.append(y)
        diffs.append(diff)

    # Convert back to full image coordinate
    diffs = np.array(diffs)
    ys = np.array(ys)
    max_diff = np.max(diffs)
    max_y = ys[np.argmax(diffs)]

    y_to_check = ys[diffs > (2500 if max_diff * 0.4 > 2500 else max_diff*0.4)]
    if debug:
        print(f"{idx}: Detected upper border at row: {max_y}, {max_diff}")

    # Visualize (optional)
    vis = cv2.cvtColor(original, cv2.COLOR_GRAY2BGR)
    cv2.line(vis, (0, max_y), (w, max_y), (0, 255, 0), 3)
    for y in y_to_check:
        cv2.line(vis, (0, y), (w, y), (255, 0, 0), 3)
    cv2.line(vis, (0, int(h*0.4)), (w, int(h*0.4)), (0, 0, 255), 3)
    # plt.imshow(vis, cmap='gray')
    # plt.show()

    return vis, y_to_check[-1]
   # keep everything from border downward

In [ ]:
# Code to change image height - 10% from the bottom and at the top hopefully to the upper part of the jawbone
debug = False
top_crop_images = []

for idx in range(len(resized_images_np)):
    original_img, metadata = resized_images_np[idx]
    h, w = original_img.shape

    original_img = original_img[0:int(h*0.9), :]

    edge_img = extract_jaw_directional_edges(original_img)
    result_img, top_y_bound = find_upper_mandible_border(idx, edge_img, original_img)

    # plt.imshow(result_img, cmap='gray')
    # plt.axis('off')
    # plt.show()

    top_crop_image = original_img[top_y_bound:, :]

    top_crop_images.append((top_crop_image, (metadata, top_y_bound)))

In [ ]:
def create_bone_mask(image):
    """
    Create a mask of the bone structure to exclude it
    """
    # Step 1: Rough threshold to get bone region
    binary = cv2.adaptiveThreshold(
        image,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=1501,  # Size of neighborhood (must be odd)
        C=10  # Constant subtracted from mean
    )
    if debug:
        plt.imshow(binary, cmap='gray')
        plt.show()


    w,h = binary.shape
    part = 0.15
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (23,23))
    binary[:, :int(w*part)] = cv2.morphologyEx(binary[:, :int(w*part)], cv2.MORPH_ERODE, kernel, iterations=6)
    binary[:, -int(w*part):] = cv2.morphologyEx(binary[:, -int(w*part):], cv2.MORPH_ERODE, kernel, iterations=6)

    if debug:
        plt.imshow(binary, cmap='gray')
        plt.show()

    # Step 2: Find the outer contour (jaw outline)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return np.ones_like(image) * 255  # No mask

    # Get largest contour (the jaw)
    jaw_contours = sorted(contours, key=cv2.contourArea, reverse=True)
    if debug:
        print([cv2.contourArea(jc) for jc in jaw_contours[:10]])
    jaw_contour = jaw_contours[0]
    # Step 3: Create jaw mask
    jaw_mask_before = np.zeros_like(image)
    max_contour = 30_000
    i=0
    chosen_contours = []
    while cv2.contourArea(jaw_contours[i]) >= max_contour:
        cv2.drawContours(jaw_mask_before, [jaw_contours[i]], -1, 255, -1)
        chosen_contours.append(jaw_contours[i])
        i+=1
    if debug:
        print(np.unique(jaw_mask_before))
        plt.imshow(jaw_mask_before, cmap='gray')
        plt.show()

    # Create a copy of the binary image for hole filling
    filled = jaw_mask_before.copy()

    # Invert to find "holes" inside the contour
    temp_inv = cv2.bitwise_not(filled)

    # Now, find pixels inside the contour that are black (holes)

    h, w = jaw_mask_before.shape

    # STEP 1: Stack contours together
    combined_points = np.vstack(chosen_contours)

    # STEP 2: Create boundary from combined points
    # Option A: Convex hull (creates outer boundary)
    boundary = cv2.convexHull(combined_points)

    if debug:
        vis = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
        for contour in chosen_contours:
            cv2.drawContours(vis, [contour], -1, (0, 255, 0), 2)
        cv2.drawContours(vis, [boundary], -1, (255, 255, 0), 3)
        plt.imshow(vis, cmap='gray')
        plt.show()

    # STEP 3: Fill the boundary
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(mask, [boundary], -1, 255, -1)  # -1 = filled

    if debug:
        plt.imshow(mask, cmap='gray')
        plt.show()

    return mask

def apply_bone_mask(image, mask):
    """
    Keep only the region inside the mask (dental arch)
    """
    return cv2.bitwise_and(image, image, mask=mask)


In [ ]:
debug = True
for idx in [test_images_idxs[0]]:
    original_img, metadata = top_crop_images[idx]
    h, w = original_img.shape

    denoised = cv2.bilateralFilter(original_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(original_img, bone_mask)
    plt.imshow(teeth_region, cmap='gray')
    plt.show()

In [ ]:
debug = False
for idx in range(len(top_crop_images)):
    print(idx)
    original_img, metadata = top_crop_images[idx]
    h, w = original_img.shape

    denoised = cv2.bilateralFilter(original_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(original_img, bone_mask)
    # plt.imshow(teeth_region, cmap='gray')
    # plt.show()

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [ ]:
debug = True
wrong_imgs_idxs = [264,276]
for idx in wrong_imgs_idxs:
    print(idx)
    original_img, metadata = top_crop_images[idx]
    h, w = original_img.shape

    denoised = cv2.bilateralFilter(original_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(original_img, bone_mask)x
    plt.imshow(teeth_region, cmap='gray')
    plt.show()
    plt.imshow(original_img, cmap='gray')
    plt.show()

In [ ]:
for idx in [test_images_idxs[3]]:
    fig, axes = plt.subplots(4,4, figsize=(32, 16))
    original_img, metadata = resized_images_np[idx]
    preprocessed = preprocess_dental_xray(original_img, axes)

    teeth_mask, contours, watershed_img = segment_teeth_watershed(original_img, preprocessed, axes, metadata)
    for ax in axes.flat:
        ax.axis('off')
    plt.show()

In [ ]:
for idx in [test_images_idxs[0]]:
    fig, axes = plt.subplots(4,4, figsize=(32, 16))
    original_img, metadata = resized_images_np[idx]
    preprocessed = preprocess_dental_xray(original_img, axes)

    teeth_mask, contours, watershed_img = segment_teeth_watershed(original_img, preprocessed, axes, metadata)
    for ax in axes.flat:
        ax.axis('off')
    plt.show()

In [ ]:
def draw_annotations(image, metadata, color=(0, 255, 0), thickness=4):
    img_color = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    for obj in metadata['objects']:
        pts = np.array(obj['points']['exterior'], dtype=np.int32)
        cv2.polylines(img_color, [pts], isClosed=True, color=color, thickness=thickness)
    return img_color

def visualize_results(original, preprocessed, teeth_mask, contours, watershed_img, gt_metadata):
    fig, axes = plt.subplots(2, 3, figsize=(24, 16))

    axes[0,0].imshow(original, cmap='gray')
    axes[0,0].set_title('Original')

    axes[0,1].imshow(preprocessed, cmap='gray')
    axes[0,1].set_title('After Preprocessing')

    axes[0,2].imshow(teeth_mask, cmap='gray')
    axes[0,2].set_title('Segmentation (Mask)')

    axes[1,0].imshow(watershed_img)
    axes[1,0].set_title('Watershed Markers')

    # Contours (green)
    overlay_yours = cv2.cvtColor(preprocessed, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(overlay_yours, contours, -1, (0, 255, 0), 5)
    axes[1,1].imshow(overlay_yours)
    axes[1,1].set_title('Contours (Green)')

    # Ground truth (red)
    overlay_gt = draw_annotations(preprocessed, gt_metadata, color=(255, 0, 0), thickness=4)
    axes[1,2].imshow(overlay_gt)
    axes[1,2].set_title('Ground Truth')

    for ax in axes.flat:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def preprocess_detect_jaw_hybrid(images):
    improved_images = []
    for image in images:
        preprocessed_img = image.copy()

        # STEP 1: Enhance dark regions
        clahe = cv2.createCLAHE(clipLimit=15.0, tileGridSize=(4,4))
        enhanced = clahe.apply(image)
        if debug:
            plt.imshow(enhanced, cmap='gray')
            plt.show()

        binary = cv2.adaptiveThreshold(
            enhanced,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            blockSize=101,  # Size of neighborhood (must be odd)
            C=0  # Constant subtracted from mean
        )
        if debug:
            plt.imshow(binary, cmap='gray')
            plt.show()

        edges = cv2.Canny(binary, threshold1=120, threshold2=150)

        if debug:
            plt.imshow(edges, cmap='gray')
            plt.show()

        improved_images.append(edges)

    return improved_images

In [ ]:
# Test on first few images
for idx in test_images_idxs[:1]:
    original_img, metadata = resized_images_np[idx]
    # preprocessed = best_preprocessing([original_img])[0]
    # preprocessed = preprocess_for_segmentation([original_img])[0]
    # preprocessed = preprocess_dental_xray([original_img])[0] # best so far
    preprocessed = preprocess_detect_jaw_hybrid([original_img])[0]

    teeth_mask, contours, watershed_img = segment_teeth_watershed(preprocessed)

    print(f"Image {idx} - Segmentation Result")
    visualize_results(original_img, preprocessed, teeth_mask, contours, watershed_img, metadata)

In [ ]:
def to_fft(img):
    # cv2 musi miec float32 na wejsciu
    dft = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)
    return np.fft.fftshift(dft) # 0hz (srednia) na srodek

def from_fft(dft_shifted):
    f_ishift = np.fft.ifftshift(dft_shifted) # 0hz z powrotem do rogu
    img_back = cv2.idft(f_ishift) # odwrotna transformata
    mag = cv2.magnitude(img_back[:,:,0], img_back[:,:,1]) # amplituda (modul zespolonej)
    cv2.normalize(mag, mag, 0, 255, cv2.NORM_MINMAX) # zeby nie wywalilo wartosci w kosmos
    return np.uint8(mag)

def get_bandpass_mask(image, r_min = 0, r_max = np.inf):
    rows, cols, _ = image.shape
    crow, ccol = rows // 2, cols // 2

    y, x = np.ogrid[:rows, :cols]
    dist = np.sqrt((x - ccol)**2 + (y - crow)**2) # pitagoras od srodka

    mask = np.zeros((rows, cols, 2), np.float32) # 2 kanaly bo liczby zespolone
    mask[(dist >= r_min) & (dist <= r_max)] = 1 # przepuszczamy pasmo
    return image * mask

def apply_gaussian_hpf(fft_img, d0):
    h, w = fft_img.shape[:2]
    cy, cx = h // 2, w // 2

    y, x = np.ogrid[:h, :w]
    dist_sq = (x - cx)**2 + (y - cy)**2 # kwadrat odleglosci (optymalizacja)

    mask = 1 - np.exp(-dist_sq / (2 * d0**2)) # Wzor Gaussa HPF

    mask = np.dstack([mask, mask]) # klonujemy na 2 kanaly (Re+Im)
    return fft_img * mask

def apply_butterworth_hpf(fft_img, d0, n):
    h, w = fft_img.shape[:2]
    cy, cx = h // 2, w // 2

    y, x = np.ogrid[:h, :w]
    dist = np.sqrt((x - cx)**2 + (y - cy)**2) # odleglosc od srodka

    # Wzor Butterwortha HPF. Epsilon 1e-5 zeby nie dzielic przez 0
    # n=1 (miekkie), n=5 (ostre), d0=promien odciecia
    mask = 1 / (1 + (d0 / (dist + 1e-5))**(2 * n))

    mask = np.dstack([mask, mask]) # 2 kanaly dla fft
    return fft_img * mask

def thresholding(image: np.ndarray, /, n: int = None, custom_thresholds: np.ndarray = None) -> np.ndarray:
    if n is not None:
        thresholds = np.linspace(0, 255, num=n)
    elif custom_thresholds is not None:
        thresholds = custom_thresholds
    else:
        raise ValueError("Either n or custom_thresholds must be provided")

    for i in range(1, len(thresholds)):
        avg = (thresholds[i] + thresholds[i-1]) // 2
        image[(image < thresholds[i]) & (image > thresholds[i-1])] = avg

    return image

def my_preprocessing(image: np.ndarray):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (69, 185))
    #* PLAYGROUND
    #! tutaj trzeba wywalic ta szczeke i taki jest caly cel
    # image = images_np[61]
    diff = 4

    # wywalic to cos z gory
    image[image >= 250] = 0.0 # usuwanie super jasnych napisow
    image = cv2.morphologyEx(image, cv2.MORPH_TOPHAT, kernel)

    # moze gauss dolno-przepustowy by tu wszedl
    # fourier_image = to_fft(image)
    # filtered_fourier = apply_gaussian_hpf(fourier_image, 2)
    # image = from_fft(filtered_fourier)

    gain = 5
    cutoff = 20

    image = image.astype(float) / 255.0

    image = 1 / (1 + np.exp(-gain * image))
    image = np.uint8(np.round(image * 255))

    image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)

    image = np.uint8(np.clip(np.int16(image) - cutoff, 0, 255))
    image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)

    # floodfill - nie wiem czy to najlepszy pomysl wszystko na czarno moze zamienic
    # diff = 0
    cv2.floodFill(image, None, seedPoint=(image.shape[0]//2, 3), newVal=0, loDiff=(diff,), upDiff=(diff,), flags=4)

    image = thresholding(image, n=10)

    return image

In [ ]:
def divide_dental_arch(preprocessed, debug=False):
    alpha = 0.5          # Intensity weight (Low = dark gap)
    beta = 0.5           # Standard Deviation weight (Low = flat gap)
    gamma = 10000.0      # Gradient weight (High = sharp transition)
    delta = 0.5          # Tilt weight (Low = prefer vertical)
    min_gap_pixels = 75  # Minimum horizontal distance between planes
    num_planes = 20      # Number of separation dividers
    y_top, y_bot = 0, 0  # Pixels to crop from top/bottom
    x_step = 10          # Horizontal search resolution
    x_top, x_bot = 250, 250 # Pixels to crop from top / bottom
    angle_range = 10     # Max tilt in degrees
    angle_steps = 10     # Angular search resolution
    eps = 1e-4

    x_deriv = cv2.Sobel(preprocessed, cv2.CV_64F, 1, 0, ksize=7)
    y_deriv = cv2.Sobel(preprocessed, cv2.CV_64F, 0, 1, ksize=7)

    # Search Space Setup
    x_positions = np.arange(x_bot, preprocessed.shape[1] - x_top, x_step)
    angles = np.radians(np.linspace(-angle_range, angle_range, angle_steps))
    y_range = np.arange(y_top, preprocessed.shape[0] - y_bot)
    y_center = y_range.mean()
    image_width = preprocessed.shape[1]

    # Calculate Node Costs
    node_costs = np.zeros((len(x_positions), len(angles)))
    for x_idx, x_pos in enumerate(x_positions):
        for angle_idx, angle in enumerate(angles):
            line_x = np.clip(np.tan(angle)*(y_range - y_center) + x_pos, 0, image_width-1).astype(int)
            gradient_intensity = np.abs(x_deriv[y_range, line_x]).mean()
            node_costs[x_idx, angle_idx] =  alpha * preprocessed[y_range, line_x].mean() + \
                                            beta * preprocessed[y_range, line_x].std() + \
                                            gamma / (gradient_intensity**2 + eps) + \
                                            delta * abs(angle)

    dp = np.full((num_planes, len(x_positions), len(angles)), np.inf)
    backpointers = np.zeros((num_planes, len(x_positions), len(angles), 2), dtype=int)

    dp[0, :len(x_positions)//4] = node_costs[:len(x_positions)//4]

    for plane in range(1, num_planes):
        for curr_x_idx, curr_x_pos in enumerate(x_positions):
            valid_prev_x_indices = np.where(curr_x_pos - x_positions[:curr_x_idx] >= min_gap_pixels)[0]

            if valid_prev_x_indices.size == 0:
                continue

            prev_plane_costs = dp[plane-1, valid_prev_x_indices]
            best_prev_x_local, best_prev_angle = np.unravel_index(
                np.argmin(prev_plane_costs),
                prev_plane_costs.shape
            )
            best_prev_x_idx = valid_prev_x_indices[best_prev_x_local]

            dp[plane, curr_x_idx, :] = prev_plane_costs[best_prev_x_local, best_prev_angle] + node_costs[curr_x_idx, :]
            backpointers[plane, curr_x_idx, :, 0] = best_prev_x_idx
            backpointers[plane, curr_x_idx, :, 1] = best_prev_angle

    curr_x_idx, curr_angle_idx = np.unravel_index(np.argmin(dp[-1]), (len(x_positions), len(angles)))

    lines = []
    for plane in range(num_planes-1, -1, -1):
        lines.append(
            (np.tan(angles[curr_angle_idx]) * (y_range - y_center) + x_positions[curr_x_idx]).astype(int)
        )

        next_x = backpointers[plane, curr_x_idx, curr_angle_idx, 0]
        next_a = backpointers[plane, curr_x_idx, curr_angle_idx, 1]
        curr_x_idx, curr_angle_idx = next_x, next_a

    lines = np.array(lines)

    x_grid = np.arange(image_width)

    label_map = np.zeros((len(y_range), image_width), dtype=np.uint8)

    for line_x in lines:
        label_map += (x_grid > line_x[:, np.newaxis])

    alpha_h, gamma_h = 1.0, 1000.0  # Weights for Intensity and Gradient
    grad_mag = np.sqrt(x_deriv**2 + y_deriv**2)

    # Focus search on the middle 1/3 of the image height to avoid jawbone
    y_start, y_end = preprocessed.shape[0]//3, 2*preprocessed.shape[0]//3
    roi_costs = alpha_h * preprocessed[y_start:y_end, :] + gamma_h / (grad_mag[y_start:y_end, :]**2 + eps)

    rows, cols = roi_costs.shape
    dp_h = np.full((rows, cols), np.inf)
    backtrack_h = np.zeros((rows, cols), dtype=int)

    dp_h[:, 0] = roi_costs[:, 0]

    for x in range(1, cols):
        for y in range(rows):
            # Constraint: Path can only move up 1, down 1, or stay at the same Y
            prev_y_range = np.arange(max(0, y-1), min(rows, y+2))
            prev_costs = dp_h[prev_y_range, x-1]

            best_prev_idx = np.argmin(prev_costs)
            dp_h[y, x] = prev_costs[best_prev_idx] + roi_costs[y, x]
            backtrack_h[y, x] = prev_y_range[best_prev_idx]

    occlusal_curve = np.zeros(cols, dtype=int)
    occlusal_curve[-1] = np.argmin(dp_h[:, -1])

    for x in range(cols-1, 0, -1):
        occlusal_curve[x-1] = backtrack_h[occlusal_curve[x], x]

    # Adjust back to global image coordinates
    occlusal_curve += y_start

    x_coords, y_coords = np.indices(preprocessed.shape)
    x_coords = x_coords[y_range, :] # dla 0 nie dziala
    y_coords = y_coords[y_range, :]
    is_lower = x_coords > occlusal_curve[y_coords]

    final_segmentation = label_map + (is_lower * 16)

    if debug:
        plt.figure(figsize=(12, 8))
        plt.plot(occlusal_curve - y_bot, color='white', linewidth=2) # The horizontal divider
        for line in lines:
            plt.plot(line, y_range - y_bot, color='cyan', linewidth=2)
        plt.imshow(preprocessed[y_range, :], cmap='gray')
        plt.imshow(final_segmentation, alpha=0.4, cmap='nipy_spectral')
        plt.axis('off')
        plt.show()

    return lines, occlusal_curve, final_segmentation, label_map

In [ ]:
def watershed_per_region(final_segmentation, preprocessed):
    unique_labels = np.unique(final_segmentation)
    all_contours = []
    all_mask = np.zeros_like(preprocessed)
    all_watershed= np.zeros_like(preprocessed)

    # y_top, y_bot = 20, 10
    for label in unique_labels:
        if label == 0:
            continue
        region_mask = (final_segmentation == label).astype(np.uint8) * 255

        roi = cv2.bitwise_and(preprocessed, preprocessed, mask=region_mask)

        # plt.figure(figsize=(12, 8))
        roi_mask, roi_contours, roi_watershed = segment_teeth_watershed(roi)
        # overlay_yours = cv2.cvtColor(preprocessed, cv2.COLOR_GRAY2BGR)
        # cv2.drawContours(overlay_yours, roi_contours, -1, (0, 255, 0), 5)
        # plt.imshow(overlay_yours)
        # plt.title('Contours (Green)')
        # plt.show()

        roi_watershed_full = np.zeros_like(preprocessed, dtype=np.uint8)
        roi_watershed_full[:] = roi_watershed

        all_mask = cv2.bitwise_or(all_mask, roi_mask)
        all_contours.extend(roi_contours)
        all_watershed = cv2.bitwise_or(all_watershed, roi_watershed_full)

    return all_mask, all_contours, all_watershed

In [ ]:
for idx in test_images_idxs:
    cropped_img, (metadata_dict, top_y_bound) = top_crop_images[idx]
    h, w = cropped_img.shape

    denoised = cv2.bilateralFilter(cropped_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(cropped_img, bone_mask)
    # plt.imshow(teeth_region, cmap='gray')
    # plt.show()

    # preprocessed = preprocess_dental_xray([teeth_region])[0]
    preprocessed = my_preprocessing(teeth_region)

    lines, occlusal_curve, final_segmentation, label_map = divide_dental_arch(preprocessed, debug=True)

    all_mask, all_contours, all_watershed = watershed_per_region(final_segmentation, preprocessed)

    # teeth_mask, contours, watershed_img = segment_teeth_watershed(preprocessed)
    # plt.show()

    print(f"Image {idx} - Segmentation Result")

    adjusted_metadata = copy.deepcopy(metadata_dict)
    for obj in adjusted_metadata['objects']:
        for pt in obj['points']['exterior']:
            pt[1] -= top_y_bound

    visualize_results(cropped_img, preprocessed, all_mask, all_contours, all_watershed, adjusted_metadata)

In [ ]:
def compare_segmentation_methods(idx, cropped_img):
    denoised = cv2.bilateralFilter(cropped_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(cropped_img, bone_mask)

    prep_my = my_preprocessing(teeth_region)
    prep_dental = preprocess_dental_xray([teeth_region])[0]

    mask_A, contours_A, ws_A = segment_teeth_watershed(prep_my)

    lines, occlusal_curve, final_seg_my, label_map = divide_dental_arch(prep_my, debug=False)
    mask_B, contours_B, ws_B = watershed_per_region(final_seg_my, prep_my)

    mask_C, contours_C, ws_C = segment_teeth_watershed(prep_dental)

    lines_D, occlusal_curve_D, final_seg_dental, label_map_D = divide_dental_arch(prep_dental, debug=False)
    mask_D, contours_D, ws_D = watershed_per_region(final_seg_dental, prep_dental)

    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    fig.suptitle(f"Comparison – photo {idx}", fontsize=16)

    overlay_A = cv2.cvtColor(prep_my, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(overlay_A, contours_A, -1, (0, 255, 0), 4)
    axes[0,0].imshow(overlay_A)
    axes[0,0].set_title("A: my_preprocessing + ordinary watershed")
    axes[0,0].axis('off')

    overlay_B = cv2.cvtColor(prep_my, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(overlay_B, contours_B, -1, (0, 255, 0), 4)
    axes[0,1].imshow(overlay_B)
    axes[0,1].set_title("B: my_preprocessing + watershed with regions")
    axes[0,1].axis('off')

    overlay_C = cv2.cvtColor(prep_dental, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(overlay_C, contours_C, -1, (0, 255, 0), 4)
    axes[1,0].imshow(overlay_C)
    axes[1,0].set_title("C: preprocess_dental_xray + ordinary watershed")
    axes[1,0].axis('off')

    overlay_D = cv2.cvtColor(prep_dental, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(overlay_D, contours_D, -1, (0, 255, 0), 4)
    axes[1,1].imshow(overlay_D)
    axes[1,1].set_title("D: preprocess_dental_xray + watershed with regions")
    axes[1,1].axis('off')

    plt.tight_layout()
    plt.show()

for idx in test_images_idxs:
    cropped_img, (metadata_dict, top_y_bound) = top_crop_images[idx]
    compare_segmentation_methods(idx, cropped_img)

In [ ]:
def create_filled_mask_from_metadata(image_shape, metadata):
    """
    Tworzy pełną maskę binarną (wypełnione zęby) na podstawie metadanych.
    """
    mask = np.zeros(image_shape, dtype=np.uint8)
    for obj in metadata['objects']:
        pts = np.array(obj['points']['exterior'], dtype=np.int32)
        cv2.fillPoly(mask, [pts], 255)
    return mask

def build_atlas_library_from_indices(images_with_metadata, selected_indices):
    """
    Tworzy bibliotekę atlasów tylko z ręcznie wybranych indeksów.
    """
    atlas_library = []

    print(f"Budowanie biblioteki atlasów z {len(selected_indices)} wybranych indeksów...")

    for idx in selected_indices:
        if idx >= len(images_with_metadata):
            print(f"Indeks {idx} poza zakresem – pomijam")
            continue

        cropped_img, (meta, top_y_bound) = top_crop_images[idx]

        adjusted_meta = copy.deepcopy(meta)
        for obj in adjusted_meta['objects']:
            for pt in obj['points']['exterior']:
                pt[1] -= top_y_bound

        # Preprocessing
        clahe = cv2.createCLAHE(clipLimit=8.0, tileGridSize=(8,8))
        img_pre = clahe.apply(cropped_img)

        # Generowanie maski ground truth
        gt_mask = create_filled_mask_from_metadata(cropped_img.shape, adjusted_meta)

        atlas_library.append({
            'image': img_pre,     # obraz po CLAHE – do dopasowywania
            'mask': gt_mask,      # idealna maska GT
            'original': cropped_img,      # oryginalny obraz
            'name': f"atlas_{idx}"
        })

    print(f"Utworzono {len(atlas_library)} atlasów.")
    return atlas_library

def register_atlas_to_target(target_img, atlas_dict):
    """
    Wariant z SIFT + affine (mniej distortion)
    """
    sift = cv2.SIFT_create()

    # Wyostrzanie (unsharp mask) – poprawia wykrywanie cech
    gaussian = cv2.GaussianBlur(target_img, (0, 0), 1.5)
    target_sharp = cv2.addWeighted(target_img, 1.5, gaussian, -0.5, 0)

    gaussian_atlas = cv2.GaussianBlur(atlas_dict['image'], (0, 0), 1.5)
    atlas_sharp = cv2.addWeighted(atlas_dict['image'], 1.5, gaussian_atlas, -0.5, 0)

    kp1, des1 = sift.detectAndCompute(atlas_sharp, None)
    kp2, des2 = sift.detectAndCompute(target_sharp, None)

    if des1 is None or des2 is None or len(des1) < 4 or len(des2) < 4:
        return None, 0, None

    # FLANN matcher dla SIFT
    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(des1, des2, k=2)

    # Ratio test dla dobrych matches
    good_matches = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    if len(good_matches) < 3:  # Affine potrzebuje min 3 punktów
        return None, 0, None

    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Affine
    M = cv2.estimateAffine2D(src_pts, dst_pts, ransacReprojThreshold=5.0)[0]

    if M is None:
        return None, 0, None

    # Score jako liczba dobrych matches
    match_score = len(good_matches)

    h, w = target_img.shape
    warped_mask = cv2.warpAffine(atlas_dict['mask'], M, (w, h))
    warped_img = cv2.warpAffine(atlas_dict['image'], M, (w, h))

    return warped_mask, match_score, warped_img

# --- 3. GŁÓWNA FUNKCJA SEGMENTACJI (MULTI-ATLAS) ---

def segment_teeth_atlas_based(target_img, atlas_library, top_k=3):
    """
    Dla danego zdjęcia, próbuje dopasować wszystkie atlasy,
    wybiera top_k najlepszych i uśrednia ich maski (Głosowanie).
    """
    h, w = target_img.shape

    # Preprocessing celu (musi być taki sam jak atlasu)
    clahe = cv2.createCLAHE(clipLimit=8.0, tileGridSize=(8,8))
    target_pre = clahe.apply(target_img)

    candidates = []

    for atlas in atlas_library:
        warped_mask, score, warped_img = register_atlas_to_target(target_pre, atlas)

        if warped_mask is not None:
            candidates.append({
                'mask': warped_mask,
                'score': score,
                'warped_img': warped_img
            })

    # Sortujemy po wyniku dopasowania (najlepsze na górze)
    candidates = sorted(candidates, key=lambda x: x['score'], reverse=True)

    if not candidates:
        print("Nie udało się dopasować żadnego atlasu!")
        return np.zeros((h, w), dtype=np.uint8)

    # GŁOSOWANIE (Label Fusion)
    # Bierzemy top K najlepszych masek i uśredniamy je
    best_candidates = candidates[:top_k]

    accumulated_mask = np.zeros((h, w), dtype=np.float32)

    total_score = 0
    for cand in best_candidates:
        # Możemy ważyć maski jakością dopasowania (score)
        weight = cand['score']
        accumulated_mask += cand['mask'].astype(np.float32) * weight
        total_score += weight

    # Normalizacja
    if total_score > 0:
        accumulated_mask /= total_score

    # Progowanie
    _, final_mask = cv2.threshold(accumulated_mask, 170, 255, cv2.THRESH_BINARY)
    final_mask = final_mask.astype(np.uint8)

    # Opcjonalnie: Morfologia na koniec, żeby wygładzić
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_OPEN, kernel)

    return final_mask, best_candidates[0]['warped_img']

In [ ]:
def preview_images(images_list, indices_to_show):
    fig, axes = plt.subplots(5, 5, figsize=(15, 9))
    axes = axes.flat

    for i, idx in enumerate(indices_to_show):
        if idx >= len(images_list):
            continue
        img, _ = images_list[idx]
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(f"idx {idx}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

indices_to_atlas_library = [4, 30, 31, 42, 48, 53, 64, 81, 106, 117, 126, 142, 153, 203, 214]
mixed_indices_to_atlas_library = [4, 30, 31, 42, 48, 53, 64, 83, 106, 117, 126, 142, 8, 203, 281, 70, 33, 112, 89, 215, 66, 9, 5, 3, 0]
preview_images(resized_images_np, mixed_indices_to_atlas_library)

In [ ]:
debug = True

cv2.setRNGSeed(42)
np.random.seed(42)

# atlas_library = build_atlas_library_from_indices(resized_images_np, mixed_indices_to_atlas_library)
atlas_library = build_atlas_library_from_indices(top_crop_images, mixed_indices_to_atlas_library)

for idx in [13, 267, 28, 63, 109, 113, 122, 135, 220, 271]:
    target_original, target_meta = resized_images_np[idx]

    # 1. Usuwanie tła
    target_img = target_original

    cropped_img, (metadata_dict, top_y_bound) = top_crop_images[idx]
    #
    # denoised = cv2.bilateralFilter(cropped_img, 1, 50, 50)
    # clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    # denoised = clahe.apply(denoised)
    #
    # bone_mask = create_bone_mask(denoised)
    # teeth_region = apply_bone_mask(cropped_img, bone_mask)

    # preprocessed = preprocess_dental_xray([target_img])[0]
    # preprocessed = my_preprocessing(teeth_region)

    print(f"--- Processing Image {idx} ---")

    # 2. Segmentacja Atlasowa
    final_mask, best_match_img = segment_teeth_atlas_based(cropped_img, atlas_library, top_k=3)

    # 3. Wyciągnięcie konturów do wizualizacji
    contours, _ = cv2.findContours(final_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 4. Wizualizacja
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))

    # 0: Oryginał
    axes[0].imshow(cropped_img, cmap='gray')
    axes[0].set_title("Target Image")

    # 1: Overlay atlasu
    overlay = cv2.addWeighted(cropped_img, 0.7, best_match_img, 0.3, 0)
    axes[1].imshow(overlay, cmap='gray')
    axes[1].set_title("Atlas Registration (Overlay)")

    # 2: Twoja segmentacja (zielone kontury)
    vis_result = cv2.cvtColor(cropped_img, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(vis_result, contours, -1, (0, 255, 0), 3)
    axes[2].imshow(vis_result)
    axes[2].set_title("Your Segmentation (Green)")

    # 3: Ground Truth (czerwone kontury)
    vis_gt = cv2.cvtColor(cropped_img, cv2.COLOR_GRAY2BGR)
    for obj in metadata_dict['objects']:
        pts = np.array(obj['points']['exterior'], dtype=np.int32)
        pts[:, 1] -= top_y_bound
        cv2.polylines(vis_gt, [pts], isClosed=True, color=(255, 0, 0), thickness=2)
    axes[3].imshow(vis_gt)
    axes[3].set_title("Ground Truth (Red)")

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
def compute_centroids_from_metadata(metadata, use_ground_truth=True):
    """
    Oblicza centroidy zębów z ground truth anotacji (polygonów)
    Zwraca dict {tooth_id: [cx, cy]}
    """
    centroids = {}
    for obj in metadata['objects']:
        points = np.array(obj['points']['exterior'])
        tooth_id = int(obj['classTitle'])

        # Centroid z moments lub bounding box
        contour = points.reshape(-1, 1, 2).astype(np.int32)
        moments = cv2.moments(contour)
        if moments['m00'] != 0:
            cx = moments['m10'] / moments['m00']
            cy = moments['m01'] / moments['m00']
        else:
            x, y, w, h = cv2.boundingRect(contour)
            cx = x + w / 2
            cy = y + h / 2

        centroids[tooth_id] = np.array([cx, cy])

    return centroids

def estimate_similarity_transform(src_points, dst_points):
    """
    Estymuje similarity transform (skala, rotacja, translacja) z src do dst
    Używa least squares
    Zwraca: scale, rotation_deg, translation (dx, dy)
    """
    if len(src_points) < 3:
        return 1.0, 0.0, (0, 0)

    # Centrowanie
    src_mean = np.mean(src_points, axis=0)
    dst_mean = np.mean(dst_points, axis=0)
    src_centered = src_points - src_mean
    dst_centered = dst_points - dst_mean

    # Skala
    src_scale = np.linalg.norm(src_centered)
    dst_scale = np.linalg.norm(dst_centered)
    scale = dst_scale / src_scale if src_scale > 0 else 1.0

    # Rotacja (SVD)
    H = src_centered.T @ dst_centered
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    rotation_rad = np.arctan2(R[1,0], R[0,0])
    rotation_deg = np.rad2deg(rotation_rad)

    # Translacja
    translation = dst_mean - scale * (R @ src_mean)

    return scale, rotation_deg, translation

def overlay_template_on_image(
    img,
    detected_centroids,
    template,
    jaw_type='upper',
    color=(255, 0, 255),
    thickness=2,
    draw_ellipse=True,
    debug=False
):
    overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if len(img.shape) == 2 else img.copy()

    template_ids = sorted(template.keys())
    template_points = np.array([template[tid]['mean'] for tid in template_ids])
    stds = np.array([np.mean(template[tid]['std']) for tid in template_ids])

    if not detected_centroids:
        # Fallback – teraz działa poprawnie
        template_width = np.max(template_points[:,0]) - np.min(template_points[:,0])
        image_width = img.shape[1]
        auto_scale = (image_width * 0.65) / template_width if template_width > 0 else 1.0

        rotation_deg = 0.0
        if jaw_type == 'upper':
            dy = img.shape[0] * 0.32   # górna szczęka wyżej
        else:
            dy = img.shape[0] * 0.68   # dolna niżej
        translation = (img.shape[1]/2, dy)
    else:
        common_ids = set(template_ids) & set(detected_centroids.keys())
        print(f"  Common points for {jaw_type}: {len(common_ids)}")  # diagnostyka

        if len(common_ids) < 3:
            print(f"  Za mało ({len(common_ids)}) – fallback")
            # return overlay_template_on_image(img, None, template, jaw_type, color, thickness, draw_ellipse, debug)

        src_points = np.array([template[tid]['mean'] for tid in common_ids])
        dst_points = np.array([detected_centroids[tid] for tid in common_ids])
        auto_scale, rotation_deg, translation = estimate_similarity_transform(src_points, dst_points)

    # Transformacja punktów template'u
    theta = np.deg2rad(rotation_deg)
    rot_matrix = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

    template_center = np.mean(template_points, axis=0)

    transformed_points = []
    transformed_stds = stds * auto_scale  # skalujemy też elipsy

    for pt, std in zip(template_points, transformed_stds):
        centered = pt - template_center
        rotated = rot_matrix @ centered
        scaled = rotated * auto_scale
        final_pt = scaled + np.array(translation)
        transformed_points.append(final_pt)

    transformed_points = np.array(transformed_points)

    # Rysowanie
    for i, (tid, point) in enumerate(zip(template_ids, transformed_points)):
        x, y = int(point[0]), int(point[1])

        if 0 <= x < img.shape[1] and 0 <= y < img.shape[0]:  # tylko na obrazie
            cv2.circle(overlay, (x, y), 8, color, -1)
            cv2.putText(overlay, str(tid), (x+12, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
            cv2.putText(overlay, str(tid), (x+10, y-2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

            if draw_ellipse:
                radius = int(3 * transformed_stds[i])  # 3 sigma
                cv2.ellipse(overlay, (x, y), (radius, radius), 0, 0, 360, color, thickness)

    if debug:
        print(f"Auto scale: {auto_scale:.2f}, Rot: {rotation_deg:.1f} deg, Trans: {translation}")

    return overlay, transformed_points, transformed_stds

def segment_with_grabcut(
    preprocessed_img,
    transformed_points,        # wszystkie centroidy
    transformed_stds,          # 1D – średnie odchylenie na ząb
    bbox_scale=1.8,
    min_contour_area=500,
    iter_count=5,
    debug=False
):
    if len(preprocessed_img.shape) == 2:
        color_img = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
    else:
        color_img = preprocessed_img.copy()

    h, w = preprocessed_img.shape[:2]
    teeth_contours = {}
    final_mask = np.zeros((h, w), dtype=np.uint8)

    for i, (pt, std) in enumerate(zip(transformed_points, transformed_stds)):
        cx, cy = int(pt[0]), int(pt[1])

        # Rozmiar bboxa – używamy średniego std jako promienia bazowego
        half_size = max(20, int(2.5 * std * bbox_scale))   # ← tu nie dzielimy na x/y

        x1 = max(0, cx - half_size)
        y1 = max(0, cy - half_size)
        x2 = min(w, cx + half_size)
        y2 = min(h, cy + half_size)

        rect = (x1, y1, x2 - x1, y2 - y1)

        if rect[2] < 40 or rect[3] < 40:
            if debug:
                print(f"Tooth {i} bbox za mały – pomijam")
            continue

        mask = np.zeros((h, w), np.uint8)
        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)

        try:
            cv2.grabCut(
                color_img,
                mask,
                rect,
                bgd_model,
                fgd_model,
                iterCount=iter_count,
                mode=cv2.GC_INIT_WITH_RECT
            )

            result_mask = np.where((mask == 2) | (mask == 0), 0, 255).astype(np.uint8)

            contours, _ = cv2.findContours(result_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if contours:
                best_contour = max(contours, key=cv2.contourArea)
                area = cv2.contourArea(best_contour)

                if area >= min_contour_area:
                    teeth_contours[i] = best_contour
                    cv2.drawContours(final_mask, [best_contour], -1, 255, -1)

                    if debug:
                        roi_overlay = color_img[y1:y2, x1:x2].copy()
                        cv2.drawContours(roi_overlay, [best_contour - np.array([[x1, y1]])], -1, (0, 255, 0), 2)
                        plt.figure(figsize=(6,6))
                        plt.imshow(cv2.cvtColor(roi_overlay, cv2.COLOR_BGR2RGB))
                        plt.title(f"Tooth {i} – area {area:.0f}")
                        plt.axis('off')
                        plt.show()

        except cv2.error as e:
            if debug:
                print(f"GrabCut error at tooth {i} ({cx},{cy}): {e}")
            continue

    if debug:
        final_vis = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        cv2.drawContours(final_vis, list(teeth_contours.values()), -1, (0, 255, 0), 3)
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(final_vis, cv2.COLOR_BGR2RGB))
        plt.title(f'GrabCut result: {len(teeth_contours)} teeth')
        plt.axis('off')
        plt.show()

    return teeth_contours, final_mask

In [ ]:
with open('upper_jaw_template.json', 'r') as f:
    upper_template = {int(k): v for k, v in json.load(f).items()}

with open('lower_jaw_template.json', 'r') as f:
    lower_template = {int(k): v for k, v in json.load(f).items()}

for idx in test_images_idxs:
    cropped_img, (metadata_dict, top_y_bound) = top_crop_images[idx]
    h, w = cropped_img.shape

    denoised = cv2.bilateralFilter(cropped_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    debug = False

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(cropped_img, bone_mask)
    # plt.imshow(teeth_region, cmap='gray')
    # plt.show()

    # preprocessed = preprocess_dental_xray([teeth_region])[0]
    preprocessed = my_preprocessing(teeth_region)

    # lines, occlusal_curve, final_segmentation, label_map = divide_dental_arch(preprocessed, debug=True)
    #
    # all_mask, all_contours, all_watershed = watershed_per_region(final_segmentation, preprocessed)

    teeth_mask, contours, watershed_img = segment_teeth_watershed(preprocessed)
    # plt.show()

    adjusted_metadata = copy.deepcopy(metadata_dict)
    for obj in adjusted_metadata['objects']:
        for pt in obj['points']['exterior']:
            pt[1] -= top_y_bound

    gt_centroids = compute_centroids_from_metadata(adjusted_metadata)

    upper_gt = {k: v for k, v in gt_centroids.items() if 1 <= k <= 16}
    lower_gt = {k: v for k, v in gt_centroids.items() if 17 <= k <= 32}

    print(f"Image {idx} | Upper GT teeth: {len(upper_gt)}, Lower GT teeth: {len(lower_gt)}")

    upper_overlay, upper_points, upper_stds = overlay_template_on_image(
        cropped_img,
        detected_centroids=upper_gt,
        template=upper_template,
        jaw_type='upper',
        color=(255, 0, 255),
        draw_ellipse=True,
        debug=True
    )

    lower_overlay, lower_points, lower_stds = overlay_template_on_image(
        cropped_img,
        detected_centroids=lower_gt,
        template=lower_template,
        jaw_type='lower',
        color=(0, 255, 255),
        draw_ellipse=True,
        debug=True
    )

    combined_overlay = cv2.addWeighted(upper_overlay, 0.65, lower_overlay, 0.65, 0)

    all_points = np.concatenate([upper_points, lower_points])
    all_stds   = np.concatenate([upper_stds,   lower_stds])

    teeth_contours, grabcut_mask = segment_with_grabcut(
        preprocessed,
        all_points,
        all_stds,
        bbox_scale=1.6,          # dostrój: 1.5–2.2
        min_contour_area=300,
        iter_count=5,
        debug=False               # włącz do testów
    )

    # ────────────────────────────────────────────────
    # Wizualizacja
    # ────────────────────────────────────────────────

    fig, axes = plt.subplots(2, 2, figsize=(20, 16))

    axes[0,0].imshow(cropped_img, cmap='gray')
    axes[0,0].set_title(f'Original cropped (img {idx})')

    axes[0,1].imshow(combined_overlay)
    axes[0,1].set_title('Template overlay')

    # Zielone kontury GrabCut
    grabcut_vis = cv2.cvtColor(preprocessed, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(grabcut_vis, list(teeth_contours.values()), -1, (0, 255, 0), 3)
    axes[1,0].imshow(grabcut_vis)
    axes[1,0].set_title('GrabCut – zielone kontury')

    gt_overlay = draw_annotations(cropped_img.copy(), adjusted_metadata,
                                 color=(255, 0, 0), thickness=3)
    axes[1,1].imshow(gt_overlay)
    axes[1,1].set_title('Ground Truth')

    for ax in axes.flat:
        ax.axis('off')

    plt.suptitle(f"GrabCut with Template ROIs – Image {idx}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
from scipy import ndimage
from skimage.segmentation import active_contour, random_walker
from skimage.filters import gaussian
from skimage.draw import ellipse
import json
import copy

# ============================================================================
# METODA 1: MARKER-BASED WATERSHED
# ============================================================================

def segment_with_marker_watershed(preprocessed_img, centroids, marker_radius=10, debug=False):
    """
    Watershed z centroidami jako markerami.

    Args:
        preprocessed_img: obraz po preprocessingu (grayscale)
        centroids: dict {tooth_id: [cx, cy]}
        marker_radius: promień markera w pikselach

    Returns:
        teeth_contours: dict {tooth_id: contour}
        markers: mapa markerów
    """
    h, w = preprocessed_img.shape

    # 1. Utwórz markery z centroidów
    markers = np.zeros((h, w), dtype=np.int32)
    tooth_to_marker = {}  # tooth_id -> marker_id

    marker_id = 1
    for tooth_id, (cx, cy) in centroids.items():
        cx, cy = int(cx), int(cy)
        if 0 <= cx < w and 0 <= cy < h:
            cv2.circle(markers, (cx, cy), marker_radius, marker_id, -1)
            tooth_to_marker[tooth_id] = marker_id
            marker_id += 1

    # 2. Tło - obszary ciemne (poza zębami)
    _, binary = cv2.threshold(preprocessed_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    background_id = marker_id

    # Erozja żeby tło nie wchodziło na zęby
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    eroded_bg = cv2.erode(255 - binary, kernel, iterations=2)
    markers[eroded_bg > 0] = background_id

    # 3. Watershed
    color_img = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
    markers_result = markers.copy()
    cv2.watershed(color_img, markers_result)

    # 4. Wyodrębnij kontury
    teeth_contours = {}
    for tooth_id, mid in tooth_to_marker.items():
        tooth_mask = (markers_result == mid).astype(np.uint8) * 255
        contours, _ = cv2.findContours(tooth_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            teeth_contours[tooth_id] = max(contours, key=cv2.contourArea)

    if debug:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(markers, cmap='nipy_spectral')
        axes[0].set_title('Initial Markers')
        axes[1].imshow(markers_result, cmap='nipy_spectral')
        axes[1].set_title('After Watershed')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
        axes[2].imshow(overlay)
        axes[2].set_title(f'Contours: {len(teeth_contours)} teeth')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours, markers_result


# ============================================================================
# METODA 2: REGION GROWING
# ============================================================================

def region_growing_single_tooth(img, seed, threshold=15, max_iterations=10000):
    """
    Region growing z pojedynczego seeda (centroidu).
    """
    h, w = img.shape
    seed_x, seed_y = int(seed[0]), int(seed[1])

    if not (0 <= seed_x < w and 0 <= seed_y < h):
        return None

    # Maska regionu
    region_mask = np.zeros((h, w), dtype=np.uint8)

    # Wartość referencyjna (średnia w okolicy seeda)
    neighborhood = img[max(0, seed_y-3):min(h, seed_y+4),
                       max(0, seed_x-3):min(w, seed_x+4)]
    ref_value = np.mean(neighborhood)

    # Kolejka punktów do sprawdzenia
    from collections import deque
    queue = deque([(seed_x, seed_y)])
    visited = set()
    visited.add((seed_x, seed_y))

    iterations = 0
    while queue and iterations < max_iterations:
        x, y = queue.popleft()
        iterations += 1

        # Sprawdź czy punkt pasuje do regionu
        if abs(float(img[y, x]) - ref_value) <= threshold:
            region_mask[y, x] = 255

            # Dodaj sąsiadów (8-connectivity)
            for dx, dy in [(-1,0), (1,0), (0,-1), (0,1),
                           (-1,-1), (-1,1), (1,-1), (1,1)]:
                nx, ny = x + dx, y + dy
                if 0 <= nx < w and 0 <= ny < h and (nx, ny) not in visited:
                    visited.add((nx, ny))
                    queue.append((nx, ny))

    return region_mask


def segment_with_region_growing(preprocessed_img, centroids, threshold=20, debug=False):
    """
    Segmentacja przez region growing z każdego centroidu.
    """
    h, w = preprocessed_img.shape
    all_masks = np.zeros((h, w), dtype=np.int32)
    teeth_contours = {}

    # Wygładzenie dla lepszego region growing
    smoothed = cv2.GaussianBlur(preprocessed_img, (5, 5), 1)

    for tooth_id, (cx, cy) in centroids.items():
        mask = region_growing_single_tooth(smoothed, (cx, cy), threshold=threshold)

        if mask is not None and np.sum(mask) > 100:  # Minimum area
            # Cleanup
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
            mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

            # Znajdź kontury
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                # Wybierz kontur zawierający centroid
                for cnt in contours:
                    if cv2.pointPolygonTest(cnt, (cx, cy), False) >= 0:
                        teeth_contours[tooth_id] = cnt
                        all_masks[mask > 0] = tooth_id
                        break

    if debug:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(all_masks, cmap='nipy_spectral')
        axes[0].set_title('Region Growing Masks')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
            M = cv2.moments(cnt)
            if M['m00'] > 0:
                cx = int(M['m10'] / M['m00'])
                cy = int(M['m01'] / M['m00'])
                cv2.putText(overlay, str(tid), (cx, cy),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)
        axes[1].imshow(overlay)
        axes[1].set_title(f'Region Growing: {len(teeth_contours)} teeth')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours, all_masks


# ============================================================================
# METODA 3: GRABCUT
# ============================================================================

def segment_with_grabcut(preprocessed_img, centroids, bbox_scale=1.5, debug=False):
    """
    GrabCut z bounding boxami wokół centroidów.
    """
    # Potrzebujemy obrazu kolorowego
    if len(preprocessed_img.shape) == 2:
        color_img = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
    else:
        color_img = preprocessed_img.copy()

    h, w = preprocessed_img.shape[:2]
    teeth_contours = {}

    # Szacunkowy rozmiar zęba
    estimated_tooth_width = w // 16  # ~16 zębów na szerokość
    estimated_tooth_height = h // 4  # ~4 "rzędy"

    for tooth_id, (cx, cy) in centroids.items():
        cx, cy = int(cx), int(cy)

        # Bounding box wokół centroidu
        half_w = int(estimated_tooth_width * bbox_scale / 2)
        half_h = int(estimated_tooth_height * bbox_scale / 2)

        x1 = max(0, cx - half_w)
        y1 = max(0, cy - half_h)
        x2 = min(w, cx + half_w)
        y2 = min(h, cy + half_h)

        rect = (x1, y1, x2 - x1, y2 - y1)

        if rect[2] < 20 or rect[3] < 20:
            continue

        # GrabCut
        mask = np.zeros((h, w), np.uint8)
        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)

        try:
            cv2.grabCut(color_img, mask, rect, bgd_model, fgd_model,
                       iterCount=5, mode=cv2.GC_INIT_WITH_RECT)

            # Wynik: 0,2 = tło, 1,3 = pierwszy plan
            result_mask = np.where((mask == 2) | (mask == 0), 0, 255).astype(np.uint8)

            # Znajdź kontury
            contours, _ = cv2.findContours(result_mask, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)

            if contours:
                # Wybierz największy kontur
                best_contour = max(contours, key=cv2.contourArea)
                if cv2.contourArea(best_contour) > 500:
                    teeth_contours[tooth_id] = best_contour

        except cv2.error:
            continue

    if debug:
        overlay = color_img.copy()
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        plt.title(f'GrabCut: {len(teeth_contours)} teeth')
        plt.axis('off')
        plt.show()

    return teeth_contours


# ============================================================================
# METODA 4: ACTIVE CONTOURS (SNAKES)
# ============================================================================

def segment_with_active_contours(preprocessed_img, centroids,
                                  initial_radius=30,
                                  alpha=0.01,  # elastyczność
                                  beta=0.1,    # gładkość
                                  gamma=0.01,  # krok czasowy
                                  debug=False):
    """
    Active Contours (Snakes) inicjalizowane okręgami wokół centroidów.
    """
    from skimage.segmentation import active_contour
    from skimage.filters import gaussian

    h, w = preprocessed_img.shape

    # Wygładzenie i normalizacja
    img_smooth = gaussian(preprocessed_img.astype(float), sigma=2)
    img_normalized = (img_smooth - img_smooth.min()) / (img_smooth.max() - img_smooth.min())

    # Gradient dla lepszego przyciągania do krawędzi
    from skimage.filters import sobel
    edge_map = sobel(img_normalized)

    teeth_contours = {}

    for tooth_id, (cx, cy) in centroids.items():
        cx, cy = float(cx), float(cy)

        # Inicjalizacja: okrąg wokół centroidu
        s = np.linspace(0, 2 * np.pi, 100)
        init_x = cx + initial_radius * np.cos(s)
        init_y = cy + initial_radius * np.sin(s)
        init_snake = np.array([init_x, init_y]).T

        # Sprawdź czy w granicach obrazu
        if (init_x.min() < 0 or init_x.max() >= w or
            init_y.min() < 0 or init_y.max() >= h):
            continue

        try:
            # Uruchom active contour
            snake = active_contour(
                edge_map,
                init_snake,
                alpha=alpha,
                beta=beta,
                gamma=gamma,
                max_num_iter=500,
                convergence=0.1
            )

            # Konwersja do formatu OpenCV contour
            contour = snake.astype(np.int32).reshape(-1, 1, 2)

            # Filtruj zbyt małe/duże kontury
            area = cv2.contourArea(contour)
            if 500 < area < 50000:
                teeth_contours[tooth_id] = contour

        except Exception as e:
            continue

    if debug:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        axes[0].imshow(edge_map, cmap='gray')
        axes[0].set_title('Edge Map (Sobel)')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
            # Rysuj też inicjalny okrąg
            cx, cy = centroids[tid]
            cv2.circle(overlay, (int(cx), int(cy)), initial_radius, (255, 0, 0), 1)

        axes[1].imshow(overlay)
        axes[1].set_title(f'Active Contours: {len(teeth_contours)} teeth\n(blue=initial, green=final)')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours


# ============================================================================
# METODA 5: RANDOM WALKER
# ============================================================================

def segment_with_random_walker(preprocessed_img, centroids, marker_radius=8, beta=130, debug=False):
    """
    Random Walker segmentation - probabilistyczna metoda.
    """
    from skimage.segmentation import random_walker

    h, w = preprocessed_img.shape

    # Normalizacja obrazu
    img_normalized = preprocessed_img.astype(float) / 255.0

    # Utwórz markery (0 = unknown, 1,2,3... = labels)
    markers = np.zeros((h, w), dtype=np.int32)
    tooth_to_label = {}

    label = 1
    for tooth_id, (cx, cy) in centroids.items():
        cx, cy = int(cx), int(cy)
        if 0 <= cx < w and 0 <= cy < h:
            cv2.circle(markers, (cx, cy), marker_radius, label, -1)
            tooth_to_label[tooth_id] = label
            label += 1

    # Marker tła
    _, binary = cv2.threshold(preprocessed_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (20, 20))
    bg_mask = cv2.erode(255 - binary, kernel, iterations=2)
    background_label = label
    markers[bg_mask > 0] = background_label

    # Random Walker
    try:
        labels = random_walker(img_normalized, markers, beta=beta, mode='bf')
    except Exception as e:
        print(f"Random Walker error: {e}")
        return {}, markers

    # Wyodrębnij kontury
    teeth_contours = {}
    for tooth_id, lbl in tooth_to_label.items():
        tooth_mask = (labels == lbl).astype(np.uint8) * 255
        contours, _ = cv2.findContours(tooth_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            teeth_contours[tooth_id] = max(contours, key=cv2.contourArea)

    if debug:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(markers, cmap='nipy_spectral')
        axes[0].set_title('Markers (seeds)')

        axes[1].imshow(labels, cmap='nipy_spectral')
        axes[1].set_title('Random Walker Result')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
        axes[2].imshow(overlay)
        axes[2].set_title(f'Contours: {len(teeth_contours)} teeth')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours, labels


# ============================================================================
# METODA 6: DISTANCE TRANSFORM + WATERSHED
# ============================================================================

def segment_with_distance_watershed(preprocessed_img, centroids, debug=False):
    """
    Ulepszona metoda watershed wykorzystująca distance transform
    i centroidy jako punkty startowe.
    """
    h, w = preprocessed_img.shape

    # 1. Binaryzacja
    _, binary = cv2.threshold(preprocessed_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 2. Cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

    # 3. Distance transform
    dist_transform = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
    dist_normalized = cv2.normalize(dist_transform, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # 4. Markery z centroidów - ale wzmocnione distance transform
    markers = np.zeros((h, w), dtype=np.int32)
    tooth_to_marker = {}

    marker_id = 1
    for tooth_id, (cx, cy) in centroids.items():
        cx, cy = int(cx), int(cy)
        if 0 <= cx < w and 0 <= cy < h:
            # Rozmiar markera proporcjonalny do distance transform
            dist_value = dist_transform[cy, cx]
            radius = max(5, int(dist_value * 0.5))
            cv2.circle(markers, (cx, cy), radius, marker_id, -1)
            tooth_to_marker[tooth_id] = marker_id
            marker_id += 1

    # 5. Tło
    background_id = marker_id
    markers[binary == 0] = background_id

    # 6. Watershed na distance transform (nie na oryginalnym obrazie!)
    # Inwersja distance transform dla watershed
    dist_inv = cv2.normalize(dist_transform.max() - dist_transform, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    dist_color = cv2.cvtColor(dist_inv, cv2.COLOR_GRAY2BGR)

    markers_result = markers.copy()
    cv2.watershed(dist_color, markers_result)

    # 7. Kontury
    teeth_contours = {}
    for tooth_id, mid in tooth_to_marker.items():
        tooth_mask = (markers_result == mid).astype(np.uint8) * 255
        contours, _ = cv2.findContours(tooth_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            teeth_contours[tooth_id] = max(contours, key=cv2.contourArea)

    if debug:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))

        axes[0,0].imshow(binary, cmap='gray')
        axes[0,0].set_title('Binary')

        axes[0,1].imshow(dist_transform, cmap='hot')
        axes[0,1].set_title('Distance Transform')

        axes[0,2].imshow(markers, cmap='nipy_spectral')
        axes[0,2].set_title('Initial Markers')

        axes[1,0].imshow(dist_inv, cmap='gray')
        axes[1,0].set_title('Inverted Distance (for watershed)')

        axes[1,1].imshow(markers_result, cmap='nipy_spectral')
        axes[1,1].set_title('After Watershed')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
        axes[1,2].imshow(overlay)
        axes[1,2].set_title(f'Result: {len(teeth_contours)} teeth')

        for ax in axes.flat:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours, markers_result


# ============================================================================
# FUNKCJA PORÓWNUJĄCA WSZYSTKIE METODY
# ============================================================================

def compare_all_segmentation_methods(preprocessed_img, centroids, gt_metadata=None):
    """
    Porównuje wszystkie metody segmentacji na jednym obrazie.
    """
    methods = {
        'Marker Watershed': lambda: segment_with_marker_watershed(preprocessed_img, centroids, debug=False),
        'Region Growing': lambda: segment_with_region_growing(preprocessed_img, centroids, debug=False),
        'GrabCut': lambda: segment_with_grabcut(preprocessed_img, centroids, debug=False),
        'Active Contours': lambda: segment_with_active_contours(preprocessed_img, centroids, debug=False),
        'Random Walker': lambda: segment_with_random_walker(preprocessed_img, centroids, debug=False),
        'Distance Watershed': lambda: segment_with_distance_watershed(preprocessed_img, centroids, debug=False),
    }

    results = {}

    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    axes = axes.flatten()

    # Oryginalny obraz z centroidami
    overlay_orig = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
    for tid, (cx, cy) in centroids.items():
        cv2.circle(overlay_orig, (int(cx), int(cy)), 5, (255, 0, 0), -1)
        cv2.putText(overlay_orig, str(tid), (int(cx)+5, int(cy)),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
    axes[0].imshow(overlay_orig)
    axes[0].set_title(f'Centroids ({len(centroids)} teeth)')
    axes[0].axis('off')

    # Ground truth jeśli dostępne
    if gt_metadata:
        gt_overlay = draw_annotations(preprocessed_img.copy(), gt_metadata,
                                      color=(255, 0, 0), thickness=2)
        axes[1].imshow(gt_overlay)
        axes[1].set_title('Ground Truth')
        axes[1].axis('off')
        start_idx = 2
    else:
        start_idx = 1

    # Każda metoda
    for i, (name, method) in enumerate(methods.items()):
        ax = axes[start_idx + i]

        try:
            result = method()
            if isinstance(result, tuple):
                contours = result[0]
            else:
                contours = result

            results[name] = contours

            overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
            for tid, cnt in contours.items():
                cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)

            ax.imshow(overlay)
            ax.set_title(f'{name}\n({len(contours)} teeth)')

        except Exception as e:
            ax.text(0.5, 0.5, f'Error:\n{str(e)[:50]}',
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{name}\n(FAILED)')
            results[name] = {}

        ax.axis('off')

    plt.tight_layout()
    plt.show()

    return results

In [ ]:
with open('upper_jaw_template.json', 'r') as f:
    upper_template = {int(k): v for k, v in json.load(f).items()}

with open('lower_jaw_template.json', 'r') as f:
    lower_template = {int(k): v for k, v in json.load(f).items()}


def get_transformed_centroids(img, gt_centroids, template, jaw_type):
    """
    Zwraca centroidy z template'u przetransformowane do współrzędnych obrazu.
    (Wykorzystuje Twoją funkcję overlay_template_on_image)
    """
    template_ids = sorted(template.keys())
    template_points = np.array([template[tid]['mean'] for tid in template_ids])

    # Znajdź wspólne punkty
    common_ids = set(template_ids) & set(gt_centroids.keys())

    if len(common_ids) >= 3:
        src_points = np.array([template[tid]['mean'] for tid in common_ids])
        dst_points = np.array([gt_centroids[tid] for tid in common_ids])
        scale, rotation_deg, translation = estimate_similarity_transform(src_points, dst_points)
    else:
        # Fallback
        template_width = np.max(template_points[:,0]) - np.min(template_points[:,0])
        image_width = img.shape[1]
        scale = (image_width * 0.65) / template_width if template_width > 0 else 1.0
        rotation_deg = 0.0
        if jaw_type == 'upper':
            translation = (img.shape[1]/2, img.shape[0] * 0.32)
        else:
            translation = (img.shape[1]/2, img.shape[0] * 0.68)

    # Transformacja
    theta = np.deg2rad(rotation_deg)
    rot_matrix = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

    template_center = np.mean(template_points, axis=0)

    transformed_centroids = {}
    for tid, pt in zip(template_ids, template_points):
        centered = pt - template_center
        rotated = rot_matrix @ centered
        scaled = rotated * scale
        final_pt = scaled + np.array(translation)
        transformed_centroids[tid] = final_pt

    return transformed_centroids


# Główna pętla
for idx in test_images_idxs[:3]:
    cropped_img, (metadata_dict, top_y_bound) = top_crop_images[idx]
    h, w = cropped_img.shape

    # Preprocessing
    denoised = cv2.bilateralFilter(cropped_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(cropped_img, bone_mask)
    preprocessed = my_preprocessing(teeth_region)

    # Ground truth centroids
    adjusted_metadata = copy.deepcopy(metadata_dict)
    for obj in adjusted_metadata['objects']:
        for pt in obj['points']['exterior']:
            pt[1] -= top_y_bound

    gt_centroids = compute_centroids_from_metadata(adjusted_metadata)

    # Rozdziel na szczęki
    upper_gt = {k: v for k, v in gt_centroids.items() if 1 <= k <= 16}
    lower_gt = {k: v for k, v in gt_centroids.items() if 17 <= k <= 32}

    # Transformuj centroidy z template'u
    upper_centroids = get_transformed_centroids(cropped_img, upper_gt, upper_template, 'upper')
    lower_centroids = get_transformed_centroids(cropped_img, lower_gt, lower_template, 'lower')

    # Połącz wszystkie centroidy
    all_centroids = {**upper_centroids, **lower_centroids}

    print(f"\n{'='*60}")
    print(f"Image {idx}: {len(all_centroids)} centroids")
    print('='*60)

    # Porównaj wszystkie metody
    results = compare_all_segmentation_methods(preprocessed, all_centroids, adjusted_metadata)

    # Podsumowanie
    print("\nResults summary:")
    for method_name, contours in results.items():
        print(f"  {method_name}: {len(contours)} teeth detected")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.spatial import Voronoi
from skimage.segmentation import (
    slic, mark_boundaries,
    morphological_geodesic_active_contour,
    morphological_chan_vese,
    flood_fill
)
from skimage.filters import gaussian, sobel
from skimage.feature import canny
from skimage.util import img_as_float
from collections import defaultdict
import json
import copy

# ============================================================================
# METODA 1: VORONOI + REFINEMENT
# ============================================================================

def segment_with_voronoi(preprocessed_img, centroids, refine=True, debug=False):
    """
    Podział Voronoi na podstawie centroidów + opcjonalne ograniczenie do zębów.

    Zalety:
    - Bardzo szybka
    - Zawsze daje wynik dla każdego centroidu
    - Naturalny podział przestrzeni
    """
    h, w = preprocessed_img.shape

    # 1. Przygotuj punkty dla Voronoi (dodaj punkty brzegowe)
    points = []
    tooth_ids = []

    for tid, (cx, cy) in centroids.items():
        if 0 <= cx < w and 0 <= cy < h:
            points.append([cx, cy])
            tooth_ids.append(tid)

    if len(points) < 3:
        return {}, np.zeros((h, w), dtype=np.int32)

    # Dodaj punkty na rogach (żeby Voronoi był skończony)
    margin = 50
    corner_points = [
        [-margin, -margin], [w + margin, -margin],
        [-margin, h + margin], [w + margin, h + margin],
        [w/2, -margin], [w/2, h + margin],
        [-margin, h/2], [w + margin, h/2]
    ]
    all_points = np.array(points + corner_points)

    # 2. Oblicz Voronoi
    vor = Voronoi(all_points)

    # 3. Utwórz mapę regionów
    # Dla każdego piksela znajdź najbliższy centroid
    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    pixel_coords = np.stack([xx.ravel(), yy.ravel()], axis=1)

    # Oblicz odległości do każdego centroidu
    centroids_arr = np.array(points)

    # Znajdź najbliższy centroid dla każdego piksela
    from scipy.spatial.distance import cdist
    distances = cdist(pixel_coords, centroids_arr)
    nearest_idx = np.argmin(distances, axis=1)

    voronoi_map = nearest_idx.reshape(h, w)

    # 4. Opcjonalne: ogranicz do obszaru zębów (binaryzacja)
    if refine:
        _, binary = cv2.threshold(preprocessed_img, 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # Cleanup
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        # Maskuj regiony Voronoi
        voronoi_map_refined = voronoi_map.copy()
        voronoi_map_refined[binary == 0] = -1  # tło
    else:
        voronoi_map_refined = voronoi_map

    # 5. Wyodrębnij kontury
    teeth_contours = {}

    for i, tid in enumerate(tooth_ids):
        tooth_mask = (voronoi_map_refined == i).astype(np.uint8) * 255

        contours, _ = cv2.findContours(tooth_mask, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            # Wybierz kontur zawierający centroid
            cx, cy = centroids[tid]
            for cnt in contours:
                if cv2.pointPolygonTest(cnt, (float(cx), float(cy)), False) >= 0:
                    teeth_contours[tid] = cnt
                    break
            else:
                # Jeśli żaden nie zawiera, weź największy
                teeth_contours[tid] = max(contours, key=cv2.contourArea)

    if debug:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(voronoi_map, cmap='nipy_spectral')
        axes[0].set_title('Voronoi Partition')

        if refine:
            axes[1].imshow(voronoi_map_refined, cmap='nipy_spectral')
            axes[1].set_title('Voronoi + Binary Mask')
        else:
            axes[1].imshow(preprocessed_img, cmap='gray')
            axes[1].set_title('Original')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
        axes[2].imshow(overlay)
        axes[2].set_title(f'Voronoi Contours: {len(teeth_contours)} teeth')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours, voronoi_map_refined


# ============================================================================
# METODA 2: SLIC SUPERPIXELS + MERGING
# ============================================================================

def segment_with_slic(preprocessed_img, centroids, n_segments=200,
                       compactness=10, debug=False):
    """
    SLIC Superpixels z późniejszym łączeniem na podstawie centroidów.

    Zalety:
    - Respektuje krawędzie
    - Regularne regiony
    - Dobre dla zaszumionych obrazów
    """
    h, w = preprocessed_img.shape

    # 1. Konwersja do float i normalizacja
    img_float = img_as_float(preprocessed_img)

    # 2. SLIC - utwórz superpixele
    # Dla grayscale musimy przekonwertować na 3-kanałowy
    img_3ch = np.stack([img_float, img_float, img_float], axis=2)

    segments = slic(img_3ch, n_segments=n_segments, compactness=compactness,
                    start_label=0, channel_axis=2)

    n_superpixels = segments.max() + 1

    # 3. Dla każdego centroidu znajdź superpixel
    centroid_to_segment = {}
    segment_to_tooth = defaultdict(list)

    for tid, (cx, cy) in centroids.items():
        cx, cy = int(cx), int(cy)
        if 0 <= cx < w and 0 <= cy < h:
            seg_id = segments[cy, cx]
            centroid_to_segment[tid] = seg_id
            segment_to_tooth[seg_id].append(tid)

    # 4. Rozszerz każdy region - łącz sąsiednie superpixele
    # które są podobne jasności

    # Oblicz średnią jasność każdego superpixela
    segment_means = ndimage.mean(preprocessed_img, segments,
                                  range(n_superpixels))

    # Dla każdego zęba, rozszerzaj region
    tooth_masks = {}

    for tid, start_seg in centroid_to_segment.items():
        # BFS - szukaj podobnych sąsiednich superpixeli
        visited = set([start_seg])
        queue = [start_seg]
        tooth_segments = [start_seg]

        ref_mean = segment_means[start_seg]
        threshold = 30  # różnica jasności

        while queue:
            current = queue.pop(0)

            # Znajdź sąsiednie superpixele
            current_mask = (segments == current)
            dilated = cv2.dilate(current_mask.astype(np.uint8),
                                np.ones((3, 3), np.uint8))

            neighbor_segments = set(segments[dilated > 0]) - visited

            for neighbor in neighbor_segments:
                visited.add(neighbor)

                # Sprawdź podobieństwo jasności
                if abs(segment_means[neighbor] - ref_mean) < threshold:
                    # Sprawdź czy ten superpixel nie należy do innego zęba
                    if neighbor not in centroid_to_segment.values() or \
                       (neighbor in segment_to_tooth and tid in segment_to_tooth[neighbor]):
                        queue.append(neighbor)
                        tooth_segments.append(neighbor)

        # Utwórz maskę
        mask = np.isin(segments, tooth_segments).astype(np.uint8) * 255
        tooth_masks[tid] = mask

    # 5. Wyodrębnij kontury
    teeth_contours = {}

    for tid, mask in tooth_masks.items():
        # Cleanup
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            cx, cy = centroids[tid]
            for cnt in contours:
                if cv2.pointPolygonTest(cnt, (float(cx), float(cy)), False) >= 0:
                    teeth_contours[tid] = cnt
                    break
            else:
                teeth_contours[tid] = max(contours, key=cv2.contourArea)

    if debug:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(mark_boundaries(img_float, segments))
        axes[0].set_title(f'SLIC Superpixels ({n_superpixels})')

        # Pokoloruj superpixele należące do zębów
        colored_segments = np.zeros((h, w, 3))
        colors = plt.cm.tab20(np.linspace(0, 1, len(centroids)))

        for i, (tid, mask) in enumerate(tooth_masks.items()):
            colored_segments[mask > 0] = colors[i % len(colors)][:3]

        axes[1].imshow(colored_segments)
        axes[1].set_title('Merged Superpixels')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
        axes[2].imshow(overlay)
        axes[2].set_title(f'SLIC Result: {len(teeth_contours)} teeth')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours, segments


# ============================================================================
# METODA 3: MORPHOLOGICAL GEODESIC ACTIVE CONTOURS (GAC)
# ============================================================================

def segment_with_morphological_gac(preprocessed_img, centroids,
                                    iterations=100, balloon=1, debug=False):
    """
    Morphological Geodesic Active Contours - szybsza wersja level-set.

    Zalety:
    - Nie wymaga rozwiązywania PDE
    - Szybsza niż klasyczne snakes
    - Dobrze radzi sobie z wklęsłymi kształtami
    """
    from skimage.segmentation import morphological_geodesic_active_contour
    from skimage.segmentation import inverse_gaussian_gradient

    h, w = preprocessed_img.shape

    # 1. Normalizacja
    img_float = img_as_float(preprocessed_img)

    # 2. Oblicz gradient (mapa krawędzi)
    gimage = inverse_gaussian_gradient(img_float, alpha=100, sigma=2)

    teeth_contours = {}

    for tid, (cx, cy) in centroids.items():
        cx, cy = int(cx), int(cy)

        if not (0 <= cx < w and 0 <= cy < h):
            continue

        # 3. Inicjalizacja - okrąg wokół centroidu
        init_level_set = np.zeros((h, w), dtype=np.float64)
        radius = 25
        rr, cc = np.ogrid[:h, :w]
        circle = (rr - cy)**2 + (cc - cx)**2 <= radius**2
        init_level_set[circle] = 1

        # 4. Uruchom GAC
        try:
            ls = morphological_geodesic_active_contour(
                gimage,
                num_iter=iterations,
                init_level_set=init_level_set,
                smoothing=1,
                balloon=balloon,
                threshold=0.7
            )

            # 5. Wyodrębnij kontur
            ls_uint8 = (ls > 0.5).astype(np.uint8) * 255
            contours, _ = cv2.findContours(ls_uint8, cv2.RETR_EXTERNAL,
                                            cv2.CHAIN_APPROX_SIMPLE)

            if contours:
                # Wybierz kontur zawierający centroid
                for cnt in contours:
                    if cv2.pointPolygonTest(cnt, (float(cx), float(cy)), False) >= 0:
                        area = cv2.contourArea(cnt)
                        if 300 < area < 50000:
                            teeth_contours[tid] = cnt
                        break

        except Exception as e:
            continue

    if debug:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(img_float, cmap='gray')
        axes[0].set_title('Original')

        axes[1].imshow(gimage, cmap='gray')
        axes[1].set_title('Inverse Gaussian Gradient')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
            cx, cy = centroids[tid]
            cv2.circle(overlay, (int(cx), int(cy)), 3, (255, 0, 0), -1)
        axes[2].imshow(overlay)
        axes[2].set_title(f'Morph GAC: {len(teeth_contours)} teeth')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours


# ============================================================================
# METODA 4: CHAN-VESE (Region-based Level Set)
# ============================================================================

def segment_with_chan_vese(preprocessed_img, centroids,
                            iterations=100, debug=False):
    """
    Chan-Vese segmentacja - nie wymaga krawędzi, bazuje na regionach.

    Zalety:
    - Działa gdy krawędzie są słabe
    - Dobre dla homogenicznych regionów
    - Matematycznie eleganckie
    """
    from skimage.segmentation import morphological_chan_vese

    h, w = preprocessed_img.shape
    img_float = img_as_float(preprocessed_img)

    teeth_contours = {}

    for tid, (cx, cy) in centroids.items():
        cx, cy = int(cx), int(cy)

        if not (0 <= cx < w and 0 <= cy < h):
            continue

        # Ogranicz obszar przetwarzania (ROI) dla szybkości
        roi_size = 80
        x1 = max(0, cx - roi_size)
        y1 = max(0, cy - roi_size)
        x2 = min(w, cx + roi_size)
        y2 = min(h, cy + roi_size)

        roi = img_float[y1:y2, x1:x2]
        roi_h, roi_w = roi.shape

        # Lokalne współrzędne centroidu
        local_cx = cx - x1
        local_cy = cy - y1

        # Inicjalizacja - okrąg
        init_level_set = np.zeros((roi_h, roi_w), dtype=np.float64)
        radius = 20
        rr, cc = np.ogrid[:roi_h, :roi_w]
        circle = (rr - local_cy)**2 + (cc - local_cx)**2 <= radius**2
        init_level_set[circle] = 1

        try:
            ls = morphological_chan_vese(
                roi,
                num_iter=iterations,
                init_level_set=init_level_set,
                smoothing=3,
                lambda1=1,
                lambda2=1
            )

            ls_uint8 = (ls > 0.5).astype(np.uint8) * 255
            contours, _ = cv2.findContours(ls_uint8, cv2.RETR_EXTERNAL,
                                            cv2.CHAIN_APPROX_SIMPLE)

            if contours:
                for cnt in contours:
                    # Przesuń kontur do globalnych współrzędnych
                    cnt_global = cnt + np.array([[[x1, y1]]])

                    if cv2.pointPolygonTest(cnt_global, (float(cx), float(cy)), False) >= 0:
                        area = cv2.contourArea(cnt_global)
                        if 300 < area < 50000:
                            teeth_contours[tid] = cnt_global
                        break

        except Exception as e:
            continue

    if debug:
        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)

        plt.figure(figsize=(12, 8))
        plt.imshow(overlay)
        plt.title(f'Chan-Vese: {len(teeth_contours)} teeth')
        plt.axis('off')
        plt.show()

    return teeth_contours


# ============================================================================
# METODA 5: FLOOD FILL
# ============================================================================

def segment_with_flood_fill(preprocessed_img, centroids, tolerance=25, debug=False):
    """
    Flood Fill z OpenCV - szybka i prosta metoda.

    Zalety:
    - Bardzo szybka
    - Prosta implementacja
    - Kontrola przez tolerancję
    """
    h, w = preprocessed_img.shape
    teeth_contours = {}

    # Przygotuj obraz z marginesem (wymagane przez floodFill)
    img_padded = cv2.copyMakeBorder(preprocessed_img, 1, 1, 1, 1,
                                     cv2.BORDER_CONSTANT, value=0)

    for tid, (cx, cy) in centroids.items():
        cx, cy = int(cx), int(cy)

        if not (0 <= cx < w and 0 <= cy < h):
            continue

        # Maska dla flood fill (musi być o 2 większa niż obraz)
        mask = np.zeros((h + 2, w + 2), np.uint8)

        # Kopia obrazu (floodFill modyfikuje obraz)
        img_copy = preprocessed_img.copy()

        # Flood fill
        seed_point = (cx, cy)
        new_val = 255
        lo_diff = tolerance  # dolna tolerancja
        up_diff = tolerance  # górna tolerancja

        try:
            _, _, mask, _ = cv2.floodFill(
                img_copy, mask, seed_point, new_val,
                loDiff=(lo_diff,), upDiff=(up_diff,),
                flags=cv2.FLOODFILL_MASK_ONLY | (255 << 8)
            )

            # Wytnij margines z maski
            result_mask = mask[1:-1, 1:-1]

            # Cleanup
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
            result_mask = cv2.morphologyEx(result_mask, cv2.MORPH_CLOSE, kernel)
            result_mask = cv2.morphologyEx(result_mask, cv2.MORPH_OPEN, kernel)

            contours, _ = cv2.findContours(result_mask, cv2.RETR_EXTERNAL,
                                            cv2.CHAIN_APPROX_SIMPLE)

            if contours:
                for cnt in contours:
                    if cv2.pointPolygonTest(cnt, (float(cx), float(cy)), False) >= 0:
                        area = cv2.contourArea(cnt)
                        if 300 < area < 50000:
                            teeth_contours[tid] = cnt
                        break

        except Exception as e:
            continue

    if debug:
        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)

        plt.figure(figsize=(12, 8))
        plt.imshow(overlay)
        plt.title(f'Flood Fill: {len(teeth_contours)} teeth')
        plt.axis('off')
        plt.show()

    return teeth_contours


# ============================================================================
# METODA 6: MEAN SHIFT SEGMENTATION
# ============================================================================

def segment_with_mean_shift(preprocessed_img, centroids,
                             sp=15, sr=30, debug=False):
    """
    Mean Shift clustering/segmentation.

    Zalety:
    - Automatycznie znajduje liczbę klastrów
    - Dobre dla regionów o podobnej teksturze
    """
    h, w = preprocessed_img.shape

    # Mean Shift wymaga obrazu kolorowego
    img_color = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)

    # 1. Mean Shift filtering
    shifted = cv2.pyrMeanShiftFiltering(img_color, sp=sp, sr=sr)

    # Konwersja z powrotem do grayscale
    shifted_gray = cv2.cvtColor(shifted, cv2.COLOR_BGR2GRAY)

    # 2. Binaryzacja
    _, binary = cv2.threshold(shifted_gray, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 3. Użyj Voronoi do podziału binarnej maski
    teeth_contours, _ = segment_with_voronoi(
        shifted_gray, centroids, refine=True, debug=False
    )

    # Alternatywnie: Connected components + przypisanie do centroidów
    if not teeth_contours:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        num_labels, labels, stats, comp_centroids = cv2.connectedComponentsWithStats(binary)

        # Przypisz komponenty do najbliższych centroidów
        for tid, (cx, cy) in centroids.items():
            cx, cy = int(cx), int(cy)
            if 0 <= cx < w and 0 <= cy < h:
                label = labels[cy, cx]
                if label > 0:
                    mask = (labels == label).astype(np.uint8) * 255
                    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                                    cv2.CHAIN_APPROX_SIMPLE)
                    if contours:
                        teeth_contours[tid] = max(contours, key=cv2.contourArea)

    if debug:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(preprocessed_img, cmap='gray')
        axes[0].set_title('Original')

        axes[1].imshow(shifted_gray, cmap='gray')
        axes[1].set_title('After Mean Shift')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
        axes[2].imshow(overlay)
        axes[2].set_title(f'Mean Shift: {len(teeth_contours)} teeth')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours, shifted_gray


# ============================================================================
# ZAKTUALIZOWANA FUNKCJA PORÓWNUJĄCA
# ============================================================================

def compare_all_segmentation_methods(preprocessed_img, centroids, gt_metadata=None):
    """
    Porównuje wszystkie metody segmentacji na jednym obrazie.
    """
    methods = {
        'Marker Watershed': lambda: segment_with_marker_watershed(preprocessed_img, centroids, debug=False),
        'Voronoi + Refine': lambda: segment_with_voronoi(preprocessed_img, centroids, debug=False),
        'SLIC Superpixels': lambda: segment_with_slic(preprocessed_img, centroids, debug=False),
        'GrabCut': lambda: segment_with_grabcut(preprocessed_img, centroids, debug=False),
        'Morph. GAC': lambda: segment_with_morphological_gac(preprocessed_img, centroids, debug=False),
        'Chan-Vese': lambda: segment_with_chan_vese(preprocessed_img, centroids, debug=False),
        'Flood Fill': lambda: segment_with_flood_fill(preprocessed_img, centroids, debug=False),
        'Mean Shift': lambda: segment_with_mean_shift(preprocessed_img, centroids, debug=False),
        'Random Walker': lambda: segment_with_random_walker(preprocessed_img, centroids, debug=False),
        'Active Contours': lambda: segment_with_active_contours(preprocessed_img, centroids, debug=False),
    }

    results = {}

    # Oblicz liczbę wierszy i kolumn
    n_methods = len(methods)
    n_extra = 2 if gt_metadata else 1  # centroids + opcjonalnie GT
    n_total = n_methods + n_extra
    n_cols = 4
    n_rows = (n_total + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 6 * n_rows))
    axes = axes.flatten()

    # Oryginalny obraz z centroidami
    overlay_orig = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
    for tid, (cx, cy) in centroids.items():
        cv2.circle(overlay_orig, (int(cx), int(cy)), 5, (255, 0, 0), -1)
        cv2.putText(overlay_orig, str(tid), (int(cx)+5, int(cy)),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
    axes[0].imshow(overlay_orig)
    axes[0].set_title(f'Centroids ({len(centroids)} teeth)')
    axes[0].axis('off')

    # Ground truth jeśli dostępne
    if gt_metadata:
        gt_overlay = draw_annotations(preprocessed_img.copy(), gt_metadata,
                                      color=(255, 0, 0), thickness=2)
        axes[1].imshow(gt_overlay)
        axes[1].set_title('Ground Truth')
        axes[1].axis('off')
        start_idx = 2
    else:
        start_idx = 1

    # Każda metoda
    for i, (name, method) in enumerate(methods.items()):
        ax_idx = start_idx + i
        if ax_idx >= len(axes):
            break

        ax = axes[ax_idx]

        try:
            result = method()
            if isinstance(result, tuple):
                contours = result[0]
            else:
                contours = result

            results[name] = contours

            overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
            for tid, cnt in contours.items():
                cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)

            ax.imshow(overlay)
            ax.set_title(f'{name}\n({len(contours)} teeth)')

        except Exception as e:
            ax.text(0.5, 0.5, f'Error:\n{str(e)[:50]}',
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{name}\n(FAILED)')
            results[name] = {}

        ax.axis('off')

    # Ukryj puste subploty
    for i in range(start_idx + len(methods), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

    return results

In [ ]:
# Główna pętla
for idx in test_images_idxs[:4]:
    cropped_img, (metadata_dict, top_y_bound) = top_crop_images[idx]
    h, w = cropped_img.shape

    # Preprocessing
    denoised = cv2.bilateralFilter(cropped_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(cropped_img, bone_mask)
    preprocessed = my_preprocessing(teeth_region)

    # Ground truth
    adjusted_metadata = copy.deepcopy(metadata_dict)
    for obj in adjusted_metadata['objects']:
        for pt in obj['points']['exterior']:
            pt[1] -= top_y_bound

    gt_centroids = compute_centroids_from_metadata(adjusted_metadata)

    upper_gt = {k: v for k, v in gt_centroids.items() if 1 <= k <= 16}
    lower_gt = {k: v for k, v in gt_centroids.items() if 17 <= k <= 32}

    upper_centroids = get_transformed_centroids(cropped_img, upper_gt, upper_template, 'upper')
    lower_centroids = get_transformed_centroids(cropped_img, lower_gt, lower_template, 'lower')

    all_centroids = {**upper_centroids, **lower_centroids}

    print(f"\n{'='*60}")
    print(f"Image {idx}: {len(all_centroids)} centroids")
    print('='*60)

    # Porównaj wszystkie metody
    results = compare_all_segmentation_methods(preprocessed, all_centroids, adjusted_metadata)

    # Podsumowanie
    print("\nResults summary:")
    for method_name, contours in results.items():
        print(f"  {method_name}: {len(contours)} teeth detected")

In [ ]:
from scipy.spatial.distance import cdist
from scipy.spatial import Voronoi

def segment_with_voronoi(preprocessed_img, centroids, refine=True, min_area=800, max_area=16000, debug=False):
    """
    Podział Voronoi na podstawie centroidów + opcjonalne ograniczenie do zębów.

    Zalety:
    - Bardzo szybka
    - Zawsze daje wynik dla każdego centroidu
    - Naturalny podział przestrzeni
    """
    h, w = preprocessed_img.shape

    # 1. Przygotuj punkty dla Voronoi (dodaj punkty brzegowe)
    points = []
    tooth_ids = []

    for tid, (cx, cy) in centroids.items():
        if 0 <= cx < w and 0 <= cy < h:
            points.append([cx, cy])
            tooth_ids.append(tid)

    if len(points) < 3:
        return {}, np.zeros((h, w), dtype=np.int32)

    # Dodaj punkty na rogach (żeby Voronoi był skończony)
    margin = 50
    corner_points = [
        [-margin, -margin], [w + margin, -margin],
        [-margin, h + margin], [w + margin, h + margin],
        [w/2, -margin], [w/2, h + margin],
        [-margin, h/2], [w + margin, h/2]
    ]
    all_points = np.array(points + corner_points)

    # 2. Oblicz Voronoi
    vor = Voronoi(all_points)

    # 3. Utwórz mapę regionów
    # Dla każdego piksela znajdź najbliższy centroid
    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    pixel_coords = np.stack([xx.ravel(), yy.ravel()], axis=1)

    # Oblicz odległości do każdego centroidu
    centroids_arr = np.array(points)

    # Znajdź najbliższy centroid dla każdego piksela
    from scipy.spatial.distance import cdist
    distances = cdist(pixel_coords, centroids_arr)
    nearest_idx = np.argmin(distances, axis=1)

    voronoi_map = nearest_idx.reshape(h, w)

    # 4. Opcjonalne: ogranicz do obszaru zębów (binaryzacja)
    if refine:
        _, binary = cv2.threshold(preprocessed_img, 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # Cleanup
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        # Maskuj regiony Voronoi
        voronoi_map_refined = voronoi_map.copy()
        voronoi_map_refined[binary == 0] = -1  # tło
    else:
        voronoi_map_refined = voronoi_map

    # 5. Wyodrębnij kontury
    teeth_contours = {}

    for i, tid in enumerate(tooth_ids):
        tooth_mask = (voronoi_map_refined == i).astype(np.uint8) * 255

        contours, _ = cv2.findContours(tooth_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            valid_contours = []

            cx, cy = int(centroids[tid][0]), int(centroids[tid][1])

            for cnt in contours:
                area = cv2.contourArea(cnt)
                if min_area <= area <= max_area:
                    if cv2.pointPolygonTest(cnt, (cx, cy), False) >= 0:
                        valid_contours.append(cnt)

            if valid_contours:
                best_cnt = max(valid_contours, key=cv2.contourArea)
                teeth_contours[tid] = best_cnt
            else:
                largest = max(contours, key=cv2.contourArea)
                area = cv2.contourArea(largest)
                if area > max_area:
                    print(f"Ząb {tid} obcięty odległością od centroidu: {area:.0f} → {max_area}")

                    # Tworzymy maskę tylko z największego konturu
                    mask_temp = np.zeros_like(tooth_mask)
                    cv2.drawContours(mask_temp, [largest], -1, 255, -1)

                    # Obliczamy odległość euklidesową od centroidu (ręcznie, bo distanceTransform jest od krawędzi)
                    yy, xx = np.mgrid[0:h, 0:w]
                    dist_from_center = np.sqrt((xx - cx)**2 + (yy - cy)**2)

                    # Tworzymy nową maskę – tylko piksele bliżej niż max_dist
                    max_dist = np.sqrt(max_area / np.pi) * 1.5  # promień koła + zapas

                    tooth_mask = np.where(
                        (mask_temp > 0) & (dist_from_center <= max_dist),
                        255,
                        0
                    ).astype(np.uint8)

                    # Ponownie znajdujemy kontury po obcięciu
                    contours, _ = cv2.findContours(tooth_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    if contours:
                        teeth_contours[tid] = max(contours, key=cv2.contourArea)
                    else:
                        # Jeśli po obcięciu nic nie zostało – zostawiamy oryginalny (rzadkie)
                        teeth_contours[tid] = largest

    if debug:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(voronoi_map, cmap='nipy_spectral')
        axes[0].set_title('Voronoi Partition')

        if refine:
            axes[1].imshow(voronoi_map_refined, cmap='nipy_spectral')
            axes[1].set_title('Voronoi + Binary Mask')
        else:
            axes[1].imshow(preprocessed_img, cmap='gray')
            axes[1].set_title('Original')

        overlay = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        for tid, cnt in teeth_contours.items():
            cv2.drawContours(overlay, [cnt], -1, (0, 255, 0), 2)
        axes[2].imshow(overlay)
        axes[2].set_title(f'Voronoi Contours: {len(teeth_contours)} teeth')

        for ax in axes:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

    return teeth_contours, voronoi_map_refined

def segment_with_mean_shift(preprocessed_img, centroids, sp=15, sr=30, debug=False):
    """
    Mean Shift clustering/segmentation.

    Zalety:
    - Automatycznie znajduje liczbę klastrów
    - Dobre dla regionów o podobnej teksturze
    """
    h, w = preprocessed_img.shape
    img_color = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
    shifted = cv2.pyrMeanShiftFiltering(img_color, sp=sp, sr=sr)
    shifted_gray = cv2.cvtColor(shifted, cv2.COLOR_BGR2GRAY)

    _, binary = cv2.threshold(shifted_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    teeth_contours, _ = segment_with_voronoi(shifted_gray, centroids, refine=True, debug=False)

    if debug:
        vis = cv2.cvtColor(preprocessed_img, cv2.COLOR_GRAY2BGR)
        cv2.drawContours(vis, list(teeth_contours.values()), -1, (0, 255, 0), 2)
        plt.figure(figsize=(10,8))
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        plt.title(f'v1 - Twoja wersja: {len(teeth_contours)} zębów')
        plt.axis('off')
        plt.show()

    return teeth_contours, shifted_gray

In [ ]:
with open('upper_jaw_template.json', 'r') as f:
    upper_template = {int(k): v for k, v in json.load(f).items()}

with open('lower_jaw_template.json', 'r') as f:
    lower_template = {int(k): v for k, v in json.load(f).items()}

images_indices = set(test_images_idxs)
# images_indices.update([46, 72, 74, 84, 98, 102, 121, 145, 156, 159, 162, 163, 165, 179, 232, 238, 245, 271, 283, 310, 321, 326, 344, 348, 386, 387, 396, 419])

# for idx in {*images_indices, *range(240, 260)}:
for idx in {*range(460, 480)}:
    cropped_img, (metadata_dict, top_y_bound) = top_crop_images[idx]
    h, w = cropped_img.shape

    denoised = cv2.bilateralFilter(cropped_img, 1, 50, 50)
    clahe = cv2.createCLAHE(clipLimit=20.0, tileGridSize=(8,8))
    denoised = clahe.apply(denoised)

    debug = False

    bone_mask = create_bone_mask(denoised)
    teeth_region = apply_bone_mask(cropped_img, bone_mask)
    # plt.imshow(teeth_region, cmap='gray')
    # plt.show()

    # preprocessed = preprocess_dental_xray([teeth_region])[0]
    preprocessed = my_preprocessing(teeth_region)

    # lines, occlusal_curve, final_segmentation, label_map = divide_dental_arch(preprocessed, debug=True)
    #
    # all_mask, all_contours, all_watershed = watershed_per_region(final_segmentation, preprocessed)

    teeth_mask, contours, watershed_img = segment_teeth_watershed(preprocessed)
    # plt.show()

    adjusted_metadata = copy.deepcopy(metadata_dict)
    for obj in adjusted_metadata['objects']:
        for pt in obj['points']['exterior']:
            pt[1] -= top_y_bound

    gt_centroids = compute_centroids_from_metadata(adjusted_metadata)

    upper_gt = {k: v for k, v in gt_centroids.items() if 1 <= k <= 16}
    lower_gt = {k: v for k, v in gt_centroids.items() if 17 <= k <= 32}

    print(f"Image {idx} | Upper GT teeth: {len(upper_gt)}, Lower GT teeth: {len(lower_gt)}")

    upper_overlay, upper_points, upper_stds = overlay_template_on_image(
        cropped_img,
        detected_centroids=upper_gt,
        template=upper_template,
        jaw_type='upper',
        color=(255, 0, 255),
        draw_ellipse=True,
        debug=True
    )

    lower_overlay, lower_points, lower_stds = overlay_template_on_image(
        cropped_img,
        detected_centroids=lower_gt,
        template=lower_template,
        jaw_type='lower',
        color=(0, 255, 255),
        draw_ellipse=True,
        debug=True
    )

    combined_overlay = cv2.addWeighted(upper_overlay, 0.65, lower_overlay, 0.65, 0)

    all_points = np.concatenate([upper_points, lower_points])
    all_stds   = np.concatenate([upper_stds,   lower_stds])

    contours, _ = segment_with_mean_shift(preprocessed, gt_centroids, sp=12, sr=30, debug=False)

    # Wizualizacja porównawcza
    fig, axes = plt.subplots(2, 2, figsize=(24, 14))

    axes[0,0].imshow(cropped_img, cmap='gray')
    axes[0,0].set_title('Original')

    axes[0,1].imshow(combined_overlay)
    axes[0,1].set_title('Template')

    vis1 = cv2.cvtColor(preprocessed, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(vis1, list(contours.values()), -1, (0, 255, 0), 2)
    axes[1,0].imshow(vis1)
    axes[1,0].set_title(f'({len(contours)} teeth)')

    gt_overlay = draw_annotations(cropped_img.copy(), adjusted_metadata, color=(255,0,0), thickness=3)
    axes[1,1].imshow(gt_overlay)
    axes[1,1].set_title('Ground Truth')

    plt.tight_layout()
    plt.show()

<h1><b>STEP5:</b> Add after segmentation processing <i>(optional)</i> </h1>

<h1><b>STEP6:</b> Extract features from segments </h1>

<h1><b>STEP7:</b> Model for tooth type classification </h1>

<h1><b>STEP8:</b> Prepare tooth data for template </h1>

<h1><b>STEP9:</b> Create template map </h1>

<h1><b>STEP10:</b> Generate summary of teeth status (whole pipeline + model for graph comparaison with template) </h1>